In [ ]:
import numpy as np
from affine import Affine
from scipy.ndimage import gaussian_filter
from rasterio.features import rasterize, geometry_mask
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import matplotlib.patches as mpatches
import os
import fiona
import rasterio
from rasterio.enums import MergeAlg
from sklearn.preprocessing import MinMaxScaler
from shapely.geometry import Point
import libpysal
import esda
import rasterstats
import warnings
from scipy.stats import gaussian_kde
from shapely import wkt
import h3
from shapely.geometry import Polygon
import math
from matplotlib.patches import Rectangle
import matplotlib.gridspec as gridspec
import matplotlib as mpl
from matplotlib.patches import Patch
import matplotlib.cm as cm
warnings.filterwarnings("ignore")

import sys
import importlib  
sys.path.insert(0, os.path.abspath("../.."))

import utils.config    
import utils.functions 
importlib.reload(utils.config)  
importlib.reload(utils.functions) 

from utils.config import *
from utils.functions import *

In [ ]:
# ─── Legs ─────────────────────────────────────────────────────────────────────
legs_GG_all          = gpd.read_parquet(f'{output_file_path_PL}legs_GG_all.parquet')
legs_GG_walk         = gpd.read_parquet(f'{output_file_path_PL}legs_GG_walk.parquet')
legs_GG_walk_regular = gpd.read_parquet(f'{output_file_path_PL}legs_GG_walk_regular.parquet')

# ─── Users ────────────────────────────────────────────────────────────────────
users_GG              = pd.read_csv(f'{output_file_path_PL}users_GG.csv')
users_GG_walk         = pd.read_csv(f'{output_file_path_PL}users_GG_walk.csv')
users_GG_walk_regular = pd.read_csv(f'{output_file_path_PL}users_GG_walk_regular.csv')

print("✓ données chargées")

In [ ]:
agglo_GG  = gpd.read_file(f'{input_file_path}/network_agreg/AGGLO_PERIMETRE_AVEC_LAC-SHP/AGGLO_PERIMETRE_AVEC_LAC.shp')
agglo_GG = agglo_GG.to_crs(operation_crs)

user_stat  = pd.read_csv(f'{input_file_path_PL}241120_user_statistics.csv')
agglo_communes = gpd.read_file(f'{input_file_path}/network_agreg/AGGLO_COMMUNES-SHP/AGGLO_COMMUNES.shp')
agglo_communes = agglo_communes.to_crs(operation_crs)

# DATA DISTRIBUTION - PER USER

## PEOPLE LOCALISATION

In [ ]:
print("── Répartition par canton d'origine ─────────────────")
print(f"Total utilisateurs uniques : {legs_GG_walk_regular['user_id_fors'].nunique()}")
print()

canton_counts = (legs_GG_walk_regular
                 .groupby("user_id_fors")["KT_home_gps"]
                 .first()
                 .value_counts(dropna=False)
                 .reset_index()
                 .rename(columns={"KT_home_gps": "canton", "count": "n_users"}))

canton_counts["pct"] = (canton_counts["n_users"] / canton_counts["n_users"].sum() * 100).round(1)

print(canton_counts.to_string(index=False))

In [ ]:
legs_GG_walk_regular['home_geometry_from_survey'].head()

In [ ]:
# ─── Get one row per user ─────────────────────────────────────────────────────
home_points = (legs_GG_walk_regular
               .groupby("user_id_fors")
               .first()
               .reset_index())

# ─── Convert string to geometry ───────────────────────────────────────────────
home_points["geometry"] = home_points["home_geometry_from_survey"].apply(
    lambda x: wkt.loads(x) if pd.notna(x) and x != "" else None
)

print(f"Total users        : {len(home_points)}")
print(f"Valid geometries   : {home_points['geometry'].notna().sum()}")
print(f"NaN geometries     : {home_points['geometry'].isna().sum()}")

# ─── Create GeoDataFrame ──────────────────────────────────────────────────────
home_gdf = gpd.GeoDataFrame(
    home_points.drop(columns=["home_geometry_from_survey"]),
    geometry="geometry",
    crs="EPSG:4326"  # coordinates are in lat/lon
)

# ─── Export to .gpkg ──────────────────────────────────────────────────────────
home_gdf.to_file(os.path.join(output_file_path_PL, "GPS_home_point.gpkg"), layer="home_points_gps", driver="GPKG")
print(f"Saved to {output_file_path_PL}")

In [ ]:
home_gdf

In [ ]:
# ─── Load Switzerland boundaries ──────────────────────────────────────────────


# Check available layers
layers = fiona.listlayers(input_file_path_PL + "swissBOUNDARIES3D_1_5_LV95_LN02.gpkg")
print("Available layers :")
for l in layers:
    print(f"  - {l}")

### H3 HEXAGONES

In [ ]:
# ─── Choose resolution ────────────────────────────────────────────────────────
resolution = 8  # between 0 (coarsest) and 15 (finest) -> Résolution infos: https://h3geo.org/docs/core-library/restable

# ─── Assign H3 cell to each home point ────────────────────────────────────────
home_gdf["h3_cell"] = home_gdf.apply(
    lambda row: h3.latlng_to_cell(
        row.geometry.y,
        row.geometry.x,
        resolution
    ) if row.geometry is not None else None,
    axis=1
)

print(f"Unique H3 cells : {home_gdf['h3_cell'].nunique()}")

# ─── Count users per H3 cell ──────────────────────────────────────────────────
h3_counts = (home_gdf
             .groupby("h3_cell")
             .size()
             .reset_index(name="n_users"))

print(f"Cells with at least 1 home : {len(h3_counts)}")

# ─── Convert H3 cell IDs to hexagon geometries ────────────────────────────────
def h3_to_polygon(h3_cell):
    coords = h3.cell_to_boundary(h3_cell)
    return Polygon([(lng, lat) for lat, lng in coords])

h3_counts["geometry"] = h3_counts["h3_cell"].apply(h3_to_polygon)
h3_gdf = gpd.GeoDataFrame(h3_counts, geometry="geometry", crs="EPSG:4326")

agglo_GG = agglo_GG.to_crs(epsg=4326)

# ─── Export ───────────────────────────────────────────────────────────────────
h3_gdf.to_file(os.path.join(output_file_path_PL, "home_density_h3.gpkg"), layer="home_density_h3", driver="GPKG")
print(f"Saved : {output_file_path_PL}")

# ─── Load Switzerland and cantons boundaries ──────────────────────────────────
switzerland = gpd.read_file(input_file_path_PL +
    "swissBOUNDARIES3D_1_5_LV95_LN02.gpkg",
    layer="tlm_landesgebiet"
).to_crs(epsg=4326)

cantons = gpd.read_file(input_file_path_PL +
    "swissBOUNDARIES3D_1_5_LV95_LN02.gpkg",
    layer="tlm_kantonsgebiet"
).to_crs(epsg=4326)

print(f"Switzerland : {len(switzerland)} features")
print(f"Cantons     : {len(cantons)} features")

# ─── Find concerned cantons ───────────────────────────────────────────────────
home_in_cantons = gpd.sjoin(
    home_gdf[home_gdf.geometry.notna()],
    cantons[["geometry", "name"]],
    how="left",
    predicate="within"
)

concerned_cantons = cantons[cantons["name"].isin(
    home_in_cantons["name"].dropna().unique()
)]

print(f"Concerned cantons : {concerned_cantons['name'].tolist()}")

In [ ]:
print(legs_GG_walk_regular.groupby('source_layer')['user_id_fors'].nunique())

In [ ]:
from IPython.display import display, HTML
display(HTML(legs_GG_walk_regular.drop(columns='geometry').head(10).to_html()))

In [ ]:
print(legs_GG_walk_regular.groupby('KT_home_gps', dropna=False)['user_id_fors'].nunique())

In [ ]:
n_cantons = len(concerned_cantons)
n_cols    = 3
n_rows    = math.ceil(n_cantons / n_cols) + 1  # +1 for overview row

fig = plt.figure(figsize=(18, 6 * n_rows))
fig.suptitle(f"Home point density — H3 resolution {resolution}",
             fontsize=14, fontweight="bold")

# ─── GridSpec : first row = overview, rest = canton zooms ─────────────────────
gs = gridspec.GridSpec(n_rows, n_cols, figure=fig)

# ─── Overview map (spans full first row) ──────────────────────────────────────
ax_overview = fig.add_subplot(gs[0, :])  # ← spans all columns

switzerland.boundary.plot(ax=ax_overview, color="grey",      linewidth=0.8, linestyle="--")
cantons.boundary.plot(    ax=ax_overview, color="lightgrey", linewidth=0.5)
agglo_GG.boundary.plot(  ax=ax_overview, color="black",     linewidth=1.5)
h3_gdf.plot(column="n_users", cmap="YlOrRd",
            edgecolor="white", linewidth=0.3,
            legend=False, ax=ax_overview, alpha=0.7)

xmin_ch, ymin_ch, xmax_ch, ymax_ch = switzerland.total_bounds
ax_overview.set_xlim(xmin_ch - 0.1, xmax_ch + 0.1)
ax_overview.set_ylim(ymin_ch - 0.1, ymax_ch + 0.1)
ax_overview.set_title("Switzerland overview", fontweight="bold")
ax_overview.set_axis_off()

# ─── Red rectangles on overview for each concerned canton ─────────────────────
for _, canton_row in concerned_cantons.iterrows():
    xmin, ymin, xmax, ymax = canton_row.geometry.bounds
    ax_overview.add_patch(Rectangle(
        (xmin - 0.02, ymin - 0.02),
        (xmax - xmin) + 0.04,
        (ymax - ymin) + 0.04,
        linewidth=1.2, edgecolor="red", facecolor="none"
    ))

# ─── One zoom per canton ──────────────────────────────────────────────────────
for i, (_, canton_row) in enumerate(concerned_cantons.iterrows()):
    row = (i // n_cols) + 1  # +1 because row 0 = overview
    col = i % n_cols

    ax = fig.add_subplot(gs[row, col])  # ← simple row/col indexing

    switzerland.boundary.plot(ax=ax, color="grey",      linewidth=0.5, linestyle="--")
    cantons.boundary.plot(    ax=ax, color="lightgrey", linewidth=0.3)
    agglo_GG.boundary.plot(  ax=ax, color="black",     linewidth=1.2)
    h3_gdf.plot(column="n_users", cmap="YlOrRd",
                edgecolor="white", linewidth=0.3,
                legend=False, ax=ax, alpha=0.7)

    xmin, ymin, xmax, ymax = canton_row.geometry.bounds
    ax.set_xlim(xmin - 0.05, xmax + 0.05)
    ax.set_ylim(ymin - 0.05, ymax + 0.05)
    ax.set_title(canton_row["name"], fontweight="bold", fontsize=10)
    ax.set_axis_off()

# ─── Single colorbar ──────────────────────────────────────────────────────────
sm = plt.cm.ScalarMappable(
    cmap="YlOrRd",
    norm=plt.Normalize(vmin=h3_gdf["n_users"].min(),
                       vmax=h3_gdf["n_users"].max())
)
plt.colorbar(sm, ax=fig.axes, label="Number of home points", shrink=0.4, pad=0.02)

plt.tight_layout()
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# FIGURE — Zoom Geneva
# ═══════════════════════════════════════════════════════════════════
fig_ge, ax_ge = plt.subplots(figsize=(8, 8))

switzerland.boundary.plot(ax=ax_ge, color="grey",      linewidth=0.8, linestyle="--")
cantons.boundary.plot(    ax=ax_ge, color="lightgrey", linewidth=0.5)
cantons[cantons["name"] == "Genève"].boundary.plot(ax=ax_ge, color="black", linewidth=1.5)
h3_gdf.plot(column="n_users", cmap="YlOrRd",
            edgecolor="white", linewidth=0.3,
            legend=False, ax=ax_ge, alpha=0.8)

xmin, ymin, xmax, ymax = cantons[cantons["name"] == "Genève"].total_bounds
ax_ge.set_xlim(xmin - 0.15, xmax + 0.15)
ax_ge.set_ylim(ymin - 0.15, ymax + 0.15)
ax_ge.set_axis_off()

sm = plt.cm.ScalarMappable(
    cmap="YlOrRd",
    norm=plt.Normalize(vmin=h3_gdf["n_users"].min(),
                       vmax=h3_gdf["n_users"].max())
)
plt.colorbar(sm, ax=ax_ge, label="Number of residents", shrink=0.6, pad=0.02)
plt.tight_layout()
plt.show()

# ═══════════════════════════════════════════════════════════════════
# FIGURE — Zoom Vaud
# ═══════════════════════════════════════════════════════════════════
fig_vd, ax_vd = plt.subplots(figsize=(8, 8))

switzerland.boundary.plot(ax=ax_vd, color="grey",      linewidth=0.8, linestyle="--")
cantons.boundary.plot(    ax=ax_vd, color="lightgrey", linewidth=0.5)
cantons[cantons["name"] == "Vaud"].boundary.plot(ax=ax_vd, color="black", linewidth=1.5)
h3_gdf.plot(column="n_users", cmap="YlOrRd",
            edgecolor="white", linewidth=0.3,
            legend=False, ax=ax_vd, alpha=0.8)

xmin, ymin, xmax, ymax = cantons[cantons["name"] == "Vaud"].total_bounds
ax_vd.set_xlim(xmin - 0.15, xmax + 0.15)
ax_vd.set_ylim(ymin - 0.15, ymax + 0.15)
ax_vd.set_axis_off()

plt.colorbar(sm, ax=ax_vd, label="Number of residents", shrink=0.6, pad=0.02)
plt.tight_layout()
plt.show()

In [ ]:
# ─── Check CRS and bounds ─────────────────────────────────────────────────────
print(f"h3_gdf_clipped CRS  : {h3_gdf.crs}")
print(f"agglo CRS          : {agglo_GG.crs}")

print(f"\nh3_gdf bounds       : {h3_gdf.total_bounds}")
print(f"agglo bounds       : {agglo_GG.total_bounds}")

In [ ]:
# ─── Reproject to EPSG:4326 ───────────────────────────────────────────────────
agglo_communes_4326 = agglo_communes.to_crs("EPSG:4326")

# ─── Track invalid geometries ─────────────────────────────────────────────────
n_invalid = home_gdf.geometry.isna().sum()
home_gdf_valid = home_gdf[home_gdf.geometry.notna()].copy()

# ─── Step 1 : spatial join with Swiss cantons ─────────────────────────────────
home_with_canton = gpd.sjoin(
    home_gdf_valid,
    cantons[["name", "geometry"]],
    how="left",
    predicate="within"
).rename(columns={"name": "canton_ch"}).drop(columns=["index_right"], errors="ignore")

# ─── Step 2 : for points not in Switzerland, join with agglo communes ─────────
not_in_switzerland = home_with_canton[home_with_canton["canton_ch"].isna()].copy()

home_with_fr = gpd.sjoin(
    not_in_switzerland,
    agglo_communes_4326[["CANTON_DEP", "geometry"]],
    how="left",
    predicate="within"
).rename(columns={"CANTON_DEP": "canton_fr"}).drop(columns=["index_right"], errors="ignore")

# ─── Step 3 : recombine ───────────────────────────────────────────────────────
home_in_switzerland = home_with_canton[home_with_canton["canton_ch"].notna()].copy()
home_in_switzerland["canton_fr"] = None

home_with_fr["canton_ch"] = None

home_final = pd.concat([home_in_switzerland, home_with_fr], ignore_index=True)

# ─── Build unified provenance column ──────────────────────────────────────────
home_final["provenance"] = home_final.apply(
    lambda row: f"{row['canton_ch']} (CH)" if pd.notna(row["canton_ch"])
    else f"{row['canton_fr']} (FR)" if pd.notna(row["canton_fr"])
    else "Unknown",
    axis=1
)

provenance_counts = home_final["provenance"].value_counts()
if n_invalid > 0:
    if "Unknown" in provenance_counts.index:
        provenance_counts["Unknown"] += n_invalid  # additionne aux Unknown existants
    else:
        provenance_counts["Unknown"] = n_invalid
provenance_counts = provenance_counts.sort_values(ascending=True)

# ─── Summary ──────────────────────────────────────────────────────────────────
print(f"Total users          : {len(home_gdf)}")
print(f"No geometry          : {n_invalid}")
print(f"Valid geometries     : {len(home_gdf_valid)}")
print(f"---")
print(f"In Switzerland (CH)  : {home_final['canton_ch'].notna().sum()}")
print(f"In France (FR)       : {home_final['canton_fr'].notna().sum()}")
print(f"Unknown              : {(home_final['provenance'] == 'Unknown').sum() + n_invalid}")
print(f"\nBy Swiss canton :\n{home_final['canton_ch'].value_counts()}")
print(f"\nBy French department :\n{home_final['canton_fr'].value_counts()}")

In [ ]:
from IPython.display import display, HTML
display(HTML(home_final.drop(columns='geometry').head(10).to_html()))

In [ ]:
print(home_final["provenance"].value_counts())

In [ ]:
# ─── Colors ───────────────────────────────────────────────────────────────────
color_ch      = "#8172B2"  # violet
color_fr      = "#937860"  # marron
color_unknown = "#C44E52"  # rouge

# ─── Reorder : Unknown at the bottom, rest sorted by value ────────────────────
special = ["Unknown"]
normal = provenance_counts.drop(labels=[l for l in special if l in provenance_counts.index])
bottom = provenance_counts[[l for l in special if l in provenance_counts.index]]
provenance_counts = pd.concat([bottom, normal.sort_values(ascending=True)])

# ─── Horizontal barplot of user provenance ────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, len(provenance_counts) * 0.5))

colors = [
    color_unknown if "Unknown" in l
    else color_fr if "(FR)" in l
    else color_ch
    for l in provenance_counts.index
]

ax.barh(provenance_counts.index, provenance_counts.values, color=colors)

# ─── Labels ───────────────────────────────────────────────────────────────────
for i, v in enumerate(provenance_counts.values):
    ax.text(v + 0.3, i, str(v), va="center", fontsize=11)

# ─── Legend ───────────────────────────────────────────────────────────────────
legend_elements = [
    Patch(facecolor=color_ch, label="Switzerland (CH)"),
    Patch(facecolor=color_fr, label="France (FR)"),
    Patch(facecolor=color_unknown, label="Unknown"),
]
ax.legend(handles=legend_elements, loc="lower right", fontsize=10)

# ─── Styling ──────────────────────────────────────────────────────────────────
ax.set_xlabel("Number of users", fontsize=12)
# ax.set_title("Home location provenance of Panel Lémanique users", fontsize=13)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# ─── Transparency ─────────────────────────────────────────────────────────────
fig.patch.set_alpha(0)
ax.patch.set_alpha(0)

plt.tight_layout()
plt.show()

## STAT ON USERS

In [ ]:
# ─── Colonnes d'intérêt ───────────────────────────────────────────────────────
columns = {
    "gdr":              "Gender",
    #"prof":             "Socio-professional situation",
    "age_fr_grouped":   "Age group",
    "income_class":  "Income",
    "has_car":          "Car owner",
    "tp_level":   "Public Transport subscription level"     
}

PALETTE = ["#4C72B0", "#DD8452", "#55A868", "#C44E52",
           "#8172B2", "#937860", "#DA8BC3", "#8C8C8C"]

# ─── 1. Résumé textuel ────────────────────────────────────────────────────────
print("=" * 55)
print("  RÉSUMÉ DES RÉPONDANT·ES — users_GG_walk_regular")
print("=" * 55)

total = len(users_GG_walk_regular)
print(f"\nTotal lignes dans le dataset : {total}\n")
print(f"(sur {len(users_GG)} utilisateurs totaux dans users_GG — {total/len(users_GG)*100:.1f}%)\n")

for col, label in columns.items():
    n_valid = users_GG_walk_regular[col].notna().sum()
    n_nan   = users_GG_walk_regular[col].isna().sum()

    print(f"── {label} ({col}) ──────────────────────")
    print(f"   Réponses  : {n_valid:>5}  ({n_valid/total*100:.1f}%)")
    print(f"   NaN       : {n_nan:>5}  ({n_nan/total*100:.1f}%)")
    print(f"   Détail (hors NaN) :")
    for val, cnt in users_GG_walk_regular[col].value_counts(dropna=True).items():
        print(f"      {str(val):<25} {cnt:>5}  ({cnt/n_valid*100:.1f}%)")
    print()

# ─── 2. Graphiques ───────────────────────────────────────────────────────────


n_cols = 3
n_rows = math.ceil(len(columns) / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 6 * n_rows), gridspec_kw={'hspace': 0.6})
axes_flat = axes.flatten()  # ← tableau 2D → liste 1D

# fig.suptitle(
#     f"Distribution of regular walkers — users_GG_walk_regular (n={total})",
#     fontsize=16, fontweight="bold", y=1.02
# )

for ax, (col, label) in zip(axes_flat, columns.items()):
    
    # ← mapping à la volée selon la colonne
    mapping = {
        'gdr'           : gender_fr_to_en,
        'prof'          : prof_fr_to_en,
        'age_fr_grouped': age_fr_to_en,
        'tp_level'      : {v: labels_all_groups[k] for k, v in tp_filters},  # {0: "No PT subscription", 1: ..., 2: ...}
    }.get(col, None)

    serie = users_GG_walk_regular[col]
    if mapping:
        serie = serie.map(mapping) 
    vc      = serie.value_counts(dropna=True)
    n_nan   = serie.isna().sum()
    n_valid = serie.notna().sum()

    sizes  = vc.values.tolist()
    #labels = [str(v) for v in vc.index.tolist()]
    labels = [str(int(v)) if isinstance(v, float) and v.is_integer() else str(v) 
          for v in vc.index.tolist()]

    if n_nan > 0:
        sizes.append(n_nan)
        labels.append("NaN")

    colors = PALETTE[:len(sizes) - (1 if n_nan > 0 else 0)]
    if n_nan > 0:
        colors = colors + ["#D3D3D3"]
    
    #colors = [CATEGORY_COLORS.get(lab, "#8C8C8C") for lab in labels]


    wedges, texts, autotexts = ax.pie(
        sizes, labels=None,
        autopct=lambda p: f"{p:.1f}%" if p > 3 else "",
        colors=colors, startangle=90,
        wedgeprops=dict(edgecolor="white", linewidth=1.5),
        pctdistance=0.75,
    )
    for at in autotexts:
        at.set_fontsize(8)

    ax.set_title(
        f"{label}\n({n_valid} respondents / {n_nan} NaN)",
        fontsize=11, fontweight="bold", pad=12
    )

    patches = [mpatches.Patch(color=colors[i], label=f"{labels[i]} ({sizes[i]})")
               for i in range(len(labels))]
    ax.legend(handles=patches, loc="lower center",
              bbox_to_anchor=(0.5, -0.28),
              fontsize=8, frameon=False, ncol=2)
    
for ax in axes_flat[len(columns):]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
print(users_GG_walk_regular['has_car'].value_counts())

In [ ]:
for lab in labels:
    print(repr(lab), "→", CATEGORY_COLORS.get(lab, "NOT FOUND"))

In [ ]:
print(users_GG_walk_regular['gdr'].value_counts())
print(users_GG_walk_regular['prof'].value_counts())
print(users_GG_walk_regular['age_fr_grouped'].value_counts())

In [ ]:
# ─── Colonnes d'intérêt ───────────────────────────────────────────────────────
columns = {
    "gdr":              "Gender",
    "prof":             "Socio-professional situation",
    "age_fr_grouped":   "Age group",
    "income_class":  "Income",
    "has_car":          "Car owner",
    "tp_level":   "PT \n Subscription"     
}

PALETTE = ["#4C72B0", "#DD8452", "#55A868", "#C44E52",
           "#8172B2", "#937860", "#DA8BC3", "#8C8C8C"]

total   = len(users_GG_walk_regular)
n_users = len(users_GG)

# ─── Graphiques — une figure par variable ────────────────────────────────────
for col, label in columns.items():

    # ← mapping à la volée
    mapping = {
        'gdr'           : gender_fr_to_en,
        'prof'          : prof_fr_to_en,
        'age_fr_grouped': age_fr_to_en,
        'tp_level'      : {v: labels_all_groups[k] for k, v in tp_filters},
    }.get(col, None)

    serie = users_GG_walk_regular[col]
    if mapping:
        serie = serie.map(mapping)

    vc      = serie.value_counts(dropna=True)
    n_nan   = serie.isna().sum()
    n_valid = serie.notna().sum()

    # Construire les séries (catégories + NaN éventuel)
    bars   = vc.values.tolist()
    cats   = [str(v) for v in vc.index.tolist()]
    if n_nan > 0:
        bars.append(n_nan)
        cats.append("NaN / missing")

    colors = PALETTE[:len(bars) - (1 if n_nan > 0 else 0)]
    if n_nan > 0:
        colors = colors + ["#D3D3D3"]

    # ── Figure ────────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(8, max(3, len(cats) * 0.65 + 1.2)))

    y_pos = range(len(cats))
    bars_plot = ax.barh(list(y_pos), bars, color=colors,
                        edgecolor="white", linewidth=1.2, height=0.6)

    # Étiquettes de valeur à droite de chaque barre
    x_max = max(bars)
    for bar, val, n_val in zip(bars_plot, bars, bars):
        pct = val / n_valid * 100 if (cats[bars.index(val)] != "NaN / missing") else val / total * 100
        ax.text(val + x_max * 0.015, bar.get_y() + bar.get_height() / 2,
                f"{val}  ({pct:.1f}%)",
                va="center", ha="left", fontsize=9)

    ax.set_yticks(list(y_pos))
    ax.set_yticklabels(cats, fontsize=10)
    ax.invert_yaxis()          # catégorie la plus fréquente en haut
    ax.set_xlabel("Number of respondents", fontsize=10)
    ax.set_xlim(0, x_max * 1.28)   # marge pour les étiquettes
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(left=False)
    ax.xaxis.grid(True, linestyle="--", alpha=0.4)
    ax.set_axisbelow(True)

    ax.set_title(
        f"{label}  —  regular walkers (users_GG_walk_regular)\n"
        f"n = {total} / {n_users} users in users_GG  "
        f"({total / n_users * 100:.1f}%)  ·  "
        f"{n_valid} valid responses, {n_nan} NaN",
        fontsize=11, fontweight="bold", pad=12, loc="left"
    )

    plt.tight_layout()
    plt.show()

In [ ]:
# ─── Colonnes d'intérêt ───────────────────────────────────────────────────────

PALETTE = ["#4C72B0", "#DD8452", "#55A868", "#C44E52",
           "#8172B2", "#937860", "#DA8BC3", "#8C8C8C"]

total   = len(users_GG_walk_regular)
n_users = len(users_GG)

# ─── Graphiques — une figure par variable ────────────────────────────────────
for col, label in columns.items():

    # ← mapping à la volée
    mapping = {
        'gdr'           : gender_fr_to_en,
        'prof'          : prof_fr_to_en,
        'age_fr_grouped': age_fr_to_en,
        'tp_level'      : {v: labels_all_groups[k] for k, v in tp_filters},
    }.get(col, None)

    serie = users_GG_walk_regular[col]
    if mapping:
        serie = serie.map(mapping)

    vc      = serie.value_counts(dropna=True)
    n_nan   = serie.isna().sum()
    n_valid = serie.notna().sum()

    sizes  = vc.values.tolist()
    labels = [str(v) for v in vc.index.tolist()]
    if n_nan > 0:
        sizes.append(n_nan)
        labels.append("NaN / missing")

    colors = PALETTE[:len(sizes) - (1 if n_nan > 0 else 0)]
    if n_nan > 0:
        colors = colors + ["#D3D3D3"]

    # ── Figure ────────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(7, 6))

    wedges, texts, autotexts = ax.pie(
        sizes, labels=None,
        autopct=lambda p: f"{p:.1f}%" if p > 3 else "",
        colors=colors, startangle=90,
        wedgeprops=dict(edgecolor="white", linewidth=1.5),
        pctdistance=0.75,
    )
    for at in autotexts:
        at.set_fontsize(9)

    ax.set_title(
        f"{label}  —  regular walkers (users_GG_walk_regular)\n"
        f"n = {total} / {n_users} users in users_GG  "
        f"({total / n_users * 100:.1f}%)  ·  "
        f"{n_valid} valid responses, {n_nan} NaN",
        fontsize=11, fontweight="bold", pad=14, loc="center"
    )

    patches = [mpatches.Patch(color=colors[i], label=f"{labels[i]} ({sizes[i]})")
               for i in range(len(labels))]
    ax.legend(handles=patches, loc="lower center",
              bbox_to_anchor=(0.5, -0.18),
              fontsize=9, frameon=False, ncol=2)

    plt.tight_layout()
    plt.show()

In [ ]:
for col, label in columns.items():

    mapping = {
        'gdr'           : gender_fr_to_en,
        'prof'          : prof_fr_to_en,
        'age_fr_grouped': age_fr_to_en,
        'tp_level'      : {v: labels_all_groups[k] for k, v in tp_filters},
    }.get(col, None)

    serie = users_GG_walk_regular[col]
    if mapping:
        serie = serie.map(mapping)

    vc      = serie.value_counts(dropna=True)
    n_nan   = serie.isna().sum()
    n_valid = serie.notna().sum()

    sizes  = vc.values.tolist()
    labels = [str(v) for v in vc.index.tolist()]
    if n_nan > 0:
        sizes.append(n_nan)
        labels.append("Missing")

    colors = PALETTE[:len(sizes) - (1 if n_nan > 0 else 0)]
    if n_nan > 0:
        colors = colors + ["#D3D3D3"]

    # ── Figure ────────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(8, 6))

    wedges, texts, autotexts = ax.pie(
        sizes,
        labels=None,
        autopct=lambda p: f"{p:.1f}%" if p > 4 else "",
        colors=colors,
        startangle=90,
        wedgeprops=dict(edgecolor="white", linewidth=2, width=0.55),  # ← donut
        pctdistance=0.75,
    )

    # % en gras et plus grands
    for at in autotexts:
        at.set_fontsize(18)
        at.set_fontweight("bold")
        at.set_color("white")

    # ── Centre du donut ───────────────────────────────────────────────────────
    ax.text(0, 0.20, label, ha="center", va="center",
            fontsize=20, fontweight="bold", color="#333333")
    ax.text(0, -0.3, f"n={n_valid}", ha="center", va="center",
            fontsize=20, fontweight="bold", color="#666666")

    # ── Titre simplifié ───────────────────────────────────────────────────────
    #ax.set_title(label, fontsize=15, fontweight="bold", pad=16)

    # ── Légende à droite ─────────────────────────────────────────────────────
    patches = [mpatches.Patch(color=colors[i], label=f"{labels[i]}  ({sizes[i]}, {sizes[i]/n_valid*100:.1f}%)")
               for i in range(len(labels))]
    ax.legend(handles=patches,
              loc="center left",
              bbox_to_anchor=(1.0, 0.5),
              fontsize=11, frameon=False,
              handlelength=1.5, handleheight=1.5)

    plt.tight_layout()
    plt.show()

In [ ]:
print("── Valeurs uniques dans gdr ──────────────────────────")
print(users_GG_walk_regular["gdr"].value_counts(dropna=False))
print(f"\nValeurs distinctes : {users_GG_walk_regular['gdr'].unique()}")

In [ ]:
print(legs_GG_walk_regular.dtypes.to_string())

### GPS ACTIVITY

In [ ]:
users_GG_walk_regular["gps_days_count"] = pd.to_numeric(
    users_GG_walk_regular["gps_days_count"], errors="coerce"
)

n_nan = users_GG_walk_regular["gps_days_count"].isna().sum()
total = len(users_GG_walk_regular)

print(f"NaN in gps_days_count : {n_nan} / {total} ({n_nan/total*100:.1f}%)")

fig, ax = plt.subplots(figsize=(10, 5))

users_GG_walk_regular["gps_days_count"].hist(
    bins=30,
    ax=ax,
    color="#4C72B0",
    edgecolor="white"
)

ax.set_title(
    f"Distribution des jours d'enregistrement GPS — users_GG_walk_regular (n={total})",
    fontweight="bold"
)
ax.set_xlabel("Nombre de jours enregistrés (gps_days_count)")
ax.set_ylabel("Nombre de répondant·es")
plt.tight_layout()
plt.show()

In [ ]:
# Convert to numeric
users_GG_walk_regular["gps_days_count"] = pd.to_numeric(users_GG_walk_regular["gps_days_count"], errors="coerce")

fig, ax = plt.subplots(figsize=(8, 4))

ax.boxplot(
    users_GG_walk_regular["gps_days_count"].dropna(),
    vert=False,
    patch_artist=True,
    boxprops=dict(facecolor="#DD8452", color="#DD8452", alpha=0.7)
)

ax.set_title("GPS recording days", fontweight="bold")
ax.set_xlabel("Number of days recorded (gps_days_count)")
ax.set_yticks([])
plt.tight_layout()
#plt.savefig("gps_days_boxplot.png", dpi=150)
plt.show()

In [ ]:
stats = users_GG_walk_regular["gps_days_count"]

print("── GPS Days Count Summary ──────────────────")
print(f"   Respondents   : {stats.notna().sum()}")
print(f"   NaN           : {stats.isna().sum()}")
print(f"   Min           : {stats.min():.0f} days")
print(f"   Max           : {stats.max():.0f} days")
print(f"   Mean          : {stats.mean():.1f} days")
print(f"   Median        : {stats.median():.1f} days")
print(f"   0 days        : {(stats == 0).sum()} people")
print(f"   1 day only    : {(stats == 1).sum()} people")
print(f"   7+ days       : {(stats >= 7).sum()} people")

In [ ]:
# ─── Convert to numeric if needed ─────────────────────────────────────────────
for col in ["gps_days_count", "track_days_count", "gps_activity_rate",
            "tracks_count", "confirmation_tracks_score"]:
    users_GG_walk_regular[col] = pd.to_numeric(users_GG_walk_regular[col], errors="coerce")
    n_nan = users_GG_walk_regular[col].isna().sum()
    print(f"{col:<30} NaN: {n_nan} / {len(users_GG_walk_regular)} ({n_nan/len(users_GG_walk_regular)*100:.1f}%)")

# ─── Print summary stats ───────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("  GPS & TRACKING QUALITY SUMMARY")
print("=" * 55)

numeric_cols = {
    "gps_days_count"           : "GPS recording days",
    "track_days_count"         : "Track recording days",
    "gps_activity_rate"        : "GPS activity rate (0-1)",
    "tracks_count"             : "Total tracks count",
    "confirmation_tracks_score": "Track confirmation score (%)",
}

for col, label in numeric_cols.items():
    data = users_GG_walk_regular[col].dropna()
    print(f"\n── {label} ({col}) ──────────────────────")
    print(f"   N valid  : {len(data):>5}  /  NaN : {users_GG_walk_regular[col].isna().sum()}")
    print(f"   Min      : {data.min():.2f}")
    print(f"   Max      : {data.max():.2f}")
    print(f"   Median   : {data.median():.2f}")
    print(f"   Mean     : {data.mean():.2f}")
    print(f"   Std      : {data.std():.2f}")

# ─── Plot ─────────────────────────────────────────────────────────────────────
PALETTE = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2"]

fig, axes = plt.subplots(5, 2, figsize=(14, 25))
fig.suptitle("GPS & Tracking Quality — Unique Walkers in Geneva",
             fontsize=14, fontweight="bold", y=1.01)

for i, (col, label) in enumerate(numeric_cols.items()):
    ax_left  = axes[i, 0]
    ax_right = axes[i, 1]
    color    = PALETTE[i]
    data     = users_GG_walk_regular[col].dropna()
    median   = data.median()
    mean     = data.mean()

    # ─── Left plot : adapted per column type ──────────────────────────────────

    # ── 1. gps_days_count & track_days_count → histogram ──
    if col in ["gps_days_count", "track_days_count"]:
        ax_left.hist(data, bins=30, color=color, edgecolor="white", alpha=0.85)
        ax_left.axvline(median, color="red",    linestyle="--", label=f"Median: {median:.0f}")
        ax_left.axvline(mean,   color="orange", linestyle="--", label=f"Mean: {mean:.0f}")
        ax_left.set_xlabel(col)
        ax_left.set_ylabel("Number of users")
        ax_left.legend(fontsize=9)

    # ── 2. gps_activity_rate → density plot ──
    elif col == "gps_activity_rate":
        kde = gaussian_kde(data)
        x   = np.linspace(0, 1, 200)
        ax_left.plot(x, kde(x), color=color, linewidth=2)
        ax_left.fill_between(x, kde(x), alpha=0.3, color=color)
        ax_left.axvline(median, color="red",    linestyle="--", label=f"Median: {median:.2f}")
        ax_left.axvline(mean,   color="orange", linestyle="--", label=f"Mean: {mean:.2f}")
        ax_left.set_xlim(0, 1)
        ax_left.set_xlabel(col)
        ax_left.set_ylabel("Density")
        ax_left.legend(fontsize=9)

    # ── 3. tracks_count → histogram with log x-axis ──
    elif col == "tracks_count":
        ax_left.hist(data, bins=30, color=color, edgecolor="white", alpha=0.85)
        ax_left.set_xscale("log")
        ax_left.axvline(median, color="red",    linestyle="--", label=f"Median: {median:.0f}")
        ax_left.axvline(mean,   color="orange", linestyle="--", label=f"Mean: {mean:.0f}")
        ax_left.set_ylabel("Number of users")
        ax_left.set_xlabel(col + " — log scale")
        ax_left.legend(fontsize=9)

    # ── 4. confirmation_tracks_score → ECDF ──
    elif col == "confirmation_tracks_score":
        data_sorted = np.sort(data)
        ecdf        = np.arange(1, len(data_sorted) + 1) / len(data_sorted)
        ax_left.plot(data_sorted, ecdf, color=color, linewidth=2)
        ax_left.axhline(0.5,    color="red",    linestyle="--", linewidth=1,
                        label=f"50% of users below {np.percentile(data, 50):.1f}%")
        ax_left.axhline(0.25,   color="orange", linestyle="--", linewidth=1,
                        label=f"25% of users below {np.percentile(data, 25):.1f}%")
        ax_left.axvline(median, color="grey",   linestyle=":",  linewidth=1,
                        label=f"Median: {median:.1f}%")
        ax_left.set_xlim(0, 100)
        ax_left.set_ylim(0, 1)
        ax_left.set_ylabel("Cumulative proportion of users")
        ax_left.set_xlabel("confirmation_tracks_score (%)")
        ax_left.legend(fontsize=8)

    ax_left.set_title(label, fontweight="bold")
    ax_left.grid(axis="y", alpha=0.3)

    # ─── Right plot : boxplot for all ─────────────────────────────────────────
    bp = ax_right.boxplot(data.values,
                          vert=True, patch_artist=True, widths=0.5,
                          medianprops=dict(color="red", linewidth=2),
                          flierprops=dict(marker="o", markersize=3, alpha=0.4))
    bp["boxes"][0].set_facecolor(color)
    bp["boxes"][0].set_alpha(0.75)
    ax_right.axhline(mean, color="orange", linestyle="--", linewidth=1.2)
    ax_right.text(1.32, median, f"Median: {median:.2f}", va="center", color="red",    fontsize=9)
    ax_right.text(1.32, mean,   f"Mean: {mean:.2f}",     va="center", color="orange", fontsize=9)
    ax_right.set_title(f"{label} — distribution", fontweight="bold")
    ax_right.set_ylabel(col)
    ax_right.set_xticks([])
    ax_right.grid(axis="y", alpha=0.3)

plt.tight_layout()
#plt.savefig("gps_tracking_quality.png", dpi=150, bbox_inches="tight")
plt.show()
#print("Chart saved as 'gps_tracking_quality.png'")

In [ ]:
# ─── Check how many users have score = 100 ────────────────────────────────────
n_perfect = (users_GG_walk_regular["confirmation_tracks_score"] == 100).sum()
n_total   = users_GG_walk_regular["confirmation_tracks_score"].notna().sum()
print(f"Users with score = 100% : {n_perfect} / {n_total} ({n_perfect/n_total*100:.1f}%)")

# ─── Distribution excluding 100% ──────────────────────────────────────────────
data_excl = users_GG_walk_regular[users_GG_walk_regular["confirmation_tracks_score"] < 100]["confirmation_tracks_score"]
print(f"Users with score < 100% : {len(data_excl)}")
print(f"Their median score      : {data_excl.median():.1f}%")

In [ ]:
data = users_GG_walk_regular["confirmation_tracks_score"].dropna()

print("── confirmation_tracks_score ────────────────────────")
print(f"  N          : {len(data)}")
print(f"  Mean       : {data.mean():.1f}%")
print(f"  Median     : {data.median():.1f}%")
print(f"  Std        : {data.std():.1f}%")
print(f"  Q25        : {data.quantile(0.25):.1f}%")
print(f"  Q75        : {data.quantile(0.75):.1f}%")
print(f"  Min        : {data.min():.1f}%")
print(f"  Max        : {data.max():.1f}%")
print(f"  NaN        : {users_GG_walk_regular['confirmation_tracks_score'].isna().sum()}")

In [ ]:
FONTSIZE_LABEL  = 14
FONTSIZE_TICK   = 14
FONTSIZE_LEGEND = 14

bin_width = 5  # ← largeur de bin en %

fig, ax = plt.subplots(figsize=(8, 5))  # ← moins large
fig.patch.set_alpha(0)
ax.patch.set_alpha(0)

ax.hist(data, bins=range(0, 102, bin_width), color="#4C72B0", edgecolor="white",
        alpha=0.85, label="Confirmation score")

ax.axvline(data.mean(),         color="orange", linestyle="--", linewidth=2, label=f"Mean: {data.mean():.1f}%")
ax.axvline(data.median(),       color="red",    linestyle="--", linewidth=2, label=f"Median: {data.median():.1f}%")
ax.axvline(data.quantile(0.25), color="green",  linestyle=":",  linewidth=2, label=f"Q25: {data.quantile(0.25):.1f}%")
ax.axvline(data.quantile(0.75), color="purple", linestyle=":",  linewidth=2, label=f"Q75: {data.quantile(0.75):.1f}%")

ax.set_xlim(0, 100)
ax.set_xlabel("Track confirmation score (%)", fontsize=FONTSIZE_LABEL)
ax.set_ylabel("Number of users", fontsize=FONTSIZE_LABEL)
ax.tick_params(axis='both', labelsize=FONTSIZE_TICK)
ax.legend(fontsize=FONTSIZE_LEGEND)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

### WALK TIMELINE

In [ ]:
# Check that all converted timestamps are in Europe/Zurich
print(legs_GG_walk_regular["started_at_local"].dt.tz)
print(legs_GG_walk_regular[["started_at", "started_at_in_timezone", "started_at_local", "hour", "time_slot"]].head(5).to_string())

In [ ]:
legs_GG_walk_regular[["legs_date", "started_at", "finished_at", 
                            "day_of_week", "time_slot", "hour", "hour_end"]].head(10)

In [ ]:
legs_GG_walk_regular.columns

In [ ]:
import matplotlib.dates as mdates
import matplotlib.ticker as ticker

# ─── Convert to datetime ───────────────────────────────────────────────────────
legs_GG_walk_regular["first_activity_date"] = pd.to_datetime(legs_GG_walk_regular["first_activity_date"])
legs_GG_walk_regular["last_activity_date"]  = pd.to_datetime(legs_GG_walk_regular["last_activity_date"])

# ─── Get one row per user with their activity period ──────────────────────────
user_periods = (legs_GG_walk_regular
                .groupby("user_id_fors")
                .agg(
                    first_date = ("first_activity_date", "first"),
                    last_date  = ("last_activity_date",  "first"),
                    n_days     = ("gps_days_count",      "first")
                )
                .reset_index()
                .sort_values("first_date"))

print(f"Total users        : {len(user_periods)}")
print(f"Survey start       : {user_periods['first_date'].min()}")
print(f"Survey end         : {user_periods['last_date'].max()}")
print(f"Median period      : {(user_periods['last_date'] - user_periods['first_date']).dt.days.median():.0f} days")

In [ ]:
FONTSIZE_TITLE  = 14
FONTSIZE_LABEL  = 14
FONTSIZE_TICK   = 14
FONTSIZE_LEGEND = 14

bin_width = 1  # ← 1 bin = 1 jour

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
fig.patch.set_alpha(0)

# ─── Chart 1 : Gantt chart ────────────────────────────────────────────────────
ax1 = axes[0]
ax1.patch.set_alpha(0)

durations = user_periods['n_days']
norm      = plt.Normalize(durations.min(), durations.max())
colors    = plt.cm.viridis(norm(durations))

for i, (_, row) in enumerate(user_periods.iterrows()):
    ax1.barh(i,
             (row["last_date"] - row["first_date"]).days,
             left=row["first_date"],
             height=0.8,
             color=colors[i],
             alpha=0.7)

survey_start = user_periods["first_date"].min()
survey_end   = user_periods["last_date"].max()

monthly_ticks = pd.date_range(
    start=survey_start.replace(day=1),
    end=survey_end,
    freq="MS"
)
all_ticks   = sorted(set(list(monthly_ticks) + [survey_start, survey_end]))
tick_labels = []
for t in all_ticks:
    if t == survey_start:
        tick_labels.append(f"▶ {t.strftime('%d %b %Y')}")
    elif t == survey_end:
        tick_labels.append(f"{t.strftime('%d %b %Y')} ◀")
    else:
        tick_labels.append(t.strftime("%b %Y"))

ax1.set_xticks(all_ticks)
ax1.set_xticklabels(tick_labels, rotation=45, ha="right", fontsize=FONTSIZE_TICK)
ax1.axvline(survey_start, color="green", linestyle="--", linewidth=1, alpha=0.7,
            label=f"Start: {survey_start.strftime('%d %b %Y')}")
ax1.axvline(survey_end,   color="red",   linestyle="--", linewidth=1, alpha=0.7,
            label=f"End: {survey_end.strftime('%d %b %Y')}")
ax1.legend(fontsize=FONTSIZE_LEGEND, loc="upper left")
ax1.set_xlim(survey_start, survey_end)
ax1.set_ylabel("Users", fontsize=FONTSIZE_LABEL)
ax1.set_yticks([])
ax1.grid(axis="x", alpha=0.3)

sm   = plt.cm.ScalarMappable(cmap="viridis", norm=norm)
cbar = plt.colorbar(sm, ax=ax1, shrink=0.8)
cbar.set_label("Active GPS days (gps_days_count)", fontsize=FONTSIZE_LABEL)
cbar.ax.tick_params(labelsize=FONTSIZE_TICK)

# ─── Chart 2 : Distribution ───────────────────────────────────────────────────
ax2 = axes[1]
ax2.patch.set_alpha(0)

bins = np.arange(
    durations.min(),
    durations.max() + bin_width,
    bin_width
)

ax2.hist(durations, bins=bins, color="#4C72B0", edgecolor="white", alpha=0.85)
ax2.axvline(durations.median(), color="red",    linestyle="--",
            label=f"Median: {durations.median():.0f} days")
ax2.axvline(durations.mean(),   color="orange", linestyle="--",
            label=f"Mean: {durations.mean():.0f} days")
ax2.axvline(durations.min(),    color="green",  linestyle=":",
            label=f"Min: {durations.min():.0f} days")
ax2.axvline(durations.max(),    color="purple", linestyle=":",
            label=f"Max: {durations.max():.0f} days")
ax2.set_xlabel("Number of active GPS days (gps_days_count)", fontsize=FONTSIZE_LABEL)
ax2.set_ylabel("Number of users", fontsize=FONTSIZE_LABEL)
ax2.tick_params(axis='both', labelsize=FONTSIZE_TICK)
ax2.legend(fontsize=FONTSIZE_LEGEND)
ax2.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

ax.scatter(users_GG_walk_regular["days_in_range"],
           users_GG_walk_regular["gps_days_count"],
           alpha=0.3, color="#4C72B0", s=20)

# Perfect line (gps_days = days_in_range = 100% activity)
max_val = users_GG_walk_regular["days_in_range"].max()
ax.plot([0, max_val], [0, max_val], color="red", linestyle="--",
        linewidth=1, label="100% GPS activity")

ax.set_title("GPS days vs Survey duration per user", fontweight="bold")
ax.set_xlabel("days_in_range")
ax.set_ylabel("gps_days_count")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
user_days = (users_GG_walk_regular
             [["user_id_fors", "days_in_range", "gps_days_count", "days_without_gps"]]
             .dropna()
             .sort_values("days_in_range"))

fig, ax = plt.subplots(figsize=(16, 6))

ax.bar(range(len(user_days)), user_days["gps_days_count"],
       color="#55A868", alpha=0.85, label="GPS days (gps_days_count)")
ax.bar(range(len(user_days)), user_days["days_without_gps"],
       bottom=user_days["gps_days_count"],
       color="#C44E52", alpha=0.85, label="Days without GPS (days_whitout_gps)")

ax.set_title("GPS days vs days without GPS per user", fontweight="bold")
ax.set_xlabel("Users (sorted by survey duration)")
ax.set_ylabel("Number of days")
ax.set_xticks([])
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import gaussian_kde

fig, ax = plt.subplots(figsize=(12, 5))

for col, label, color in [
    ("days_in_range",   "Days in range",    "#4C72B0"),
    ("gps_days_count",  "GPS days",         "#55A868"),
    ("days_without_gps","Days without GPS", "#C44E52"),
]:
    data = pd.to_numeric(users_GG_walk_regular[col], errors="coerce").dropna()
    kde  = gaussian_kde(data)
    x    = np.linspace(data.min(), data.max(), 300)
    ax.plot(x, kde(x), color=color, linewidth=2, label=label)
    ax.fill_between(x, kde(x), alpha=0.15, color=color)

ax.set_title("Distribution comparison — survey days", fontweight="bold")
ax.set_xlabel("Number of days (days_in_range)")
ax.set_ylabel("Density")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ─── Get one row per user with n_days_GG ──────────────────────────────────────
user_days_GG = (legs_GG_walk_regular
                .groupby("user_id_fors")
                .agg(
                    n_days_GG  = ("n_days_GG",  "first"),
                    first_date = ("first_activity_date", "first"),
                    last_date  = ("last_activity_date",  "first"),
                )
                .reset_index()
                .sort_values("first_date"))

print(f"Total users        : {len(user_days_GG)}")
print(f"Médiane n_days_GG  : {user_days_GG['n_days_GG'].median():.0f} jours")
print(f"Moyenne n_days_GG  : {user_days_GG['n_days_GG'].mean():.0f} jours")
print(f"Min n_days_GG      : {user_days_GG['n_days_GG'].min():.0f} jours")
print(f"Max n_days_GG      : {user_days_GG['n_days_GG'].max():.0f} jours")

In [ ]:
FONTSIZE_TITLE  = 14
FONTSIZE_LABEL  = 14
FONTSIZE_TICK   = 14
FONTSIZE_LEGEND = 14

bin_width = 1  # ← 1 bin = 1 jour

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
fig.patch.set_alpha(0)

# ─── Chart 1 : Gantt chart ────────────────────────────────────────────────────
ax1 = axes[0]
ax1.patch.set_alpha(0)

norm   = plt.Normalize(user_days_GG["n_days_GG"].min(), user_days_GG["n_days_GG"].max())
colors = plt.cm.viridis(norm(user_days_GG["n_days_GG"]))

for i, (_, row) in enumerate(user_days_GG.iterrows()):
    ax1.barh(i,
             (row["last_date"] - row["first_date"]).days,
             left=row["first_date"],
             height=0.8,
             color=colors[i],
             alpha=0.7)

survey_start = user_days_GG["first_date"].min()
survey_end   = user_days_GG["last_date"].max()

monthly_ticks = pd.date_range(
    start=survey_start.replace(day=1),
    end=survey_end,
    freq="MS"
)
all_ticks   = sorted(set(list(monthly_ticks) + [survey_start, survey_end]))
tick_labels = []
for t in all_ticks:
    if t == survey_start:
        tick_labels.append(f"▶ {t.strftime('%d %b %Y')}")
    elif t == survey_end:
        tick_labels.append(f"{t.strftime('%d %b %Y')} ◀")
    else:
        tick_labels.append(t.strftime("%b %Y"))

ax1.set_xticks(all_ticks)
ax1.set_xticklabels(tick_labels, rotation=45, ha="right", fontsize=FONTSIZE_TICK)
ax1.axvline(survey_start, color="green", linestyle="--", linewidth=1, alpha=0.7,
            label=f"Start: {survey_start.strftime('%d %b %Y')}")
ax1.axvline(survey_end,   color="red",   linestyle="--", linewidth=1, alpha=0.7,
            label=f"End: {survey_end.strftime('%d %b %Y')}")
ax1.legend(fontsize=FONTSIZE_LEGEND, loc="upper left")
ax1.set_xlim(survey_start, survey_end)
ax1.set_ylabel("Users", fontsize=FONTSIZE_LABEL)
ax1.set_yticks([])
ax1.grid(axis="x", alpha=0.3)

sm   = plt.cm.ScalarMappable(cmap="viridis", norm=norm)
cbar = plt.colorbar(sm, ax=ax1, shrink=0.8)
cbar.set_label("Days in Geneva (n_days_GG)", fontsize=FONTSIZE_LABEL)
cbar.ax.tick_params(labelsize=FONTSIZE_TICK)

# ─── Chart 2 : Distribution ───────────────────────────────────────────────────
ax2 = axes[1]
ax2.patch.set_alpha(0)

bins = np.arange(
    user_days_GG["n_days_GG"].min(),
    user_days_GG["n_days_GG"].max() + bin_width,
    bin_width
)

ax2.hist(user_days_GG["n_days_GG"], bins=bins, color="#DD8452", edgecolor="white", alpha=0.85)
ax2.axvline(user_days_GG["n_days_GG"].median(), color="red",    linestyle="--",
            label=f"Median: {user_days_GG['n_days_GG'].median():.0f} days")
ax2.axvline(user_days_GG["n_days_GG"].mean(),   color="orange", linestyle="--",
            label=f"Mean: {user_days_GG['n_days_GG'].mean():.0f} days")
ax2.axvline(user_days_GG["n_days_GG"].min(),    color="green",  linestyle=":",
            label=f"Min: {user_days_GG['n_days_GG'].min():.0f} days")
ax2.axvline(user_days_GG["n_days_GG"].max(),    color="purple", linestyle=":",
            label=f"Max: {user_days_GG['n_days_GG'].max():.0f} days")
ax2.set_xlabel("Days in Geneva (n_days_GG)", fontsize=FONTSIZE_LABEL)
ax2.set_ylabel("Number of users", fontsize=FONTSIZE_LABEL)
ax2.tick_params(axis='both', labelsize=FONTSIZE_TICK)
ax2.legend(fontsize=FONTSIZE_LEGEND)
ax2.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ─── Compter les users actifs par jour ───────────────────────────────────────
min_active_users = 0.01 * len(users_GG_walk_regular)  # ← paramètre à visualiser

daily_active = (
    legs_GG_walk_regular.groupby("legs_date")["user_id_fors"]
    .nunique()
    .reset_index()
    .rename(columns={"user_id_fors": "n_active"})
    .sort_values("legs_date")
)

n_days_total    = len(daily_active)
n_days_selected = (daily_active["n_active"] >= min_active_users).sum()
n_days_excluded = n_days_total - n_days_selected

mean_active   = daily_active["n_active"].mean()
median_active = daily_active["n_active"].median()
p10_active    = daily_active["n_active"].quantile(0.10)
p25_active    = daily_active["n_active"].quantile(0.25)
p1_active     = daily_active["n_active"].quantile(0.01)

print(f"Jours totaux             : {n_days_total}")
print(f"Jours sélectionnés (>={min_active_users:.0f} users) : {n_days_selected} ({n_days_selected/n_days_total*100:.1f}%)")
print(f"Jours exclus             : {n_days_excluded} ({n_days_excluded/n_days_total*100:.1f}%)")
print(f"\nMoyenne users/jour       : {mean_active:.0f}")
print(f"Médiane users/jour       : {median_active:.0f}")
print(f"P10 users/jour           : {p10_active:.0f}")
print(f"P25 users/jour           : {p25_active:.0f}")
print(f"P1 users/jour           : {p1_active:.0f}")

# ─── Plot ─────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))

ax.bar(daily_active["legs_date"], daily_active["n_active"],
       color=["#4C72B0" if n >= min_active_users else "#DD8452"
              for n in daily_active["n_active"]],
       alpha=0.85, width=1)

# ─── Lignes de référence ──────────────────────────────────────────────────────
ax.axhline(min_active_users, color="red",    linestyle="--", linewidth=1.5,
           label=f"Seuil actuel (1% = {min_active_users:.0f} users)")
ax.axhline(mean_active,      color="green",  linestyle="-",  linewidth=1.2,
           label=f"Moyenne = {mean_active:.0f} users")
ax.axhline(median_active,    color="orange", linestyle="-",  linewidth=1.2,
           label=f"Médiane = {median_active:.0f} users")
ax.axhline(p10_active,       color="purple", linestyle=":",  linewidth=1.2,
           label=f"P10 = {p10_active:.0f} users")
ax.axhline(p25_active,       color="brown",  linestyle=":",  linewidth=1.2,
           label=f"P25 = {p25_active:.0f} users")
ax.axhline(p1_active,       color="red",  linestyle=":",  linewidth=1.2,
           label=f"P1 = {p1_active:.0f} users")

ax.set_title(
    f"Users actifs par jour — seuil à {min_active_users:.0f} users (1% panel)\n"
    f"{n_days_selected}/{n_days_total} jours sélectionnés ({n_days_selected/n_days_total*100:.1f}%) | "
    f"moyenne={mean_active:.0f} | médiane={median_active:.0f}",
    fontweight="bold"
)
ax.set_xlabel("Date")
ax.set_ylabel("Nombre d'users actifs")
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

### WALK SLOT PER DAY

In [ ]:
# ─── Raw counts per time slot ─────────────────────────────────────────────────
time_counts_raw = (legs_GG_walk_regular["time_slot"]
                   .value_counts()
                   .sort_index())

# ─── Day order and palette ────────────────────────────────────────────────────
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
PALETTE   = ["#4C72B0", "#DD8452", "#55A868", "#C44E52",
             "#8172B2", "#937860", "#DA8BC3", "#8C8C8C"]

# ─── Normalisation for chart 2 and 3 ─────────────────────────────────────────
n_occurrences_per_day = (legs_GG_walk_regular
                          .groupby("day_of_week")["legs_date"]
                          .nunique()
                          .reindex(day_order))



FONTSIZE_TITLE  = 12
FONTSIZE_LABEL  = 12
FONTSIZE_TICK   = 12
FONTSIZE_LEGEND = 12
FONTSIZE_ANNOT  = 12

# ─── Chart 1 ─────────────────────────────────────────────────────────────────
fig1, ax1 = plt.subplots(figsize=(14, 5))
fig1.patch.set_alpha(0)
ax1.patch.set_alpha(0)

ax1.bar(range(len(time_counts_raw)), time_counts_raw.values,
        color="#1f77b4", alpha=0.8, width=0.8)

tick_positions = [i for i, t in enumerate(time_counts_raw.index) if t.endswith(":00")]
tick_labels    = [t for t in time_counts_raw.index if t.endswith(":00")]
ax1.set_xticks(tick_positions)
ax1.set_xticklabels(tick_labels, rotation=45, ha="right", fontsize=FONTSIZE_TICK)
ax1.tick_params(axis='y', labelsize=FONTSIZE_TICK)

index_list = list(time_counts_raw.index)
for time_label in ["07:00", "09:00", "11:30", "14:00", "16:00", "19:00"]:
    if time_label in index_list:
        x_pos = index_list.index(time_label)
        ax1.axvline(x_pos, color="red", linestyle="--", linewidth=2, alpha=0.6)
        # ax1.text(x_pos + 0.2, ax1.get_ylim()[1] * 0.85, time_label,
        #          ha="left", fontsize=FONTSIZE_ANNOT, color="red", alpha=0.7, fontweight='bold')

ax1.set_xlabel("Departure time", fontsize=FONTSIZE_LABEL)
ax1.set_ylabel("Number of walking legs \n (total survey period)", fontsize=FONTSIZE_LABEL)
ax1.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()
plt.close()

# ─── Chart 2 ─────────────────────────────────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(10, 5))
fig2.patch.set_alpha(0)
ax2.patch.set_alpha(0)

day_counts = (legs_GG_walk_regular["day_of_week"]
              .value_counts()
              .reindex(day_order)
              .fillna(0))
day_counts_normalised = day_counts / n_occurrences_per_day

bars = ax2.bar(day_order, day_counts_normalised.values,
               color=PALETTE[:7], alpha=0.85, edgecolor="white", linewidth=1.2)

for bar, val in zip(bars, day_counts_normalised.values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             f"{val:.1f}", ha="center", va="bottom", fontsize=FONTSIZE_ANNOT)

ax2.axvspan(4.5, 6.5, alpha=0.08, color="orange")
ax2.text(5.5, ax2.get_ylim()[1] * 0.9, "Weekend",
         ha="center", fontsize=FONTSIZE_ANNOT, color="orange", alpha=0.8)

ax2.set_xlabel("Day of week", fontsize=FONTSIZE_LABEL)
ax2.set_ylabel("Average number of walks per day", fontsize=FONTSIZE_LABEL)
ax2.tick_params(axis='both', labelsize=FONTSIZE_TICK)
ax2.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()
plt.close()

# ─── Chart 3 ─────────────────────────────────────────────────────────────────
fig3, ax3 = plt.subplots(figsize=(14, 5))
fig3.patch.set_alpha(0)
ax3.patch.set_alpha(0)

heatmap_data = (legs_GG_walk_regular
                .groupby(["day_of_week", "hour"])
                .size()
                .unstack(fill_value=0)
                .reindex(day_order))
heatmap_data_normalised = heatmap_data.div(n_occurrences_per_day, axis=0)

im = ax3.imshow(heatmap_data_normalised.values, aspect="auto", cmap="YlOrRd", interpolation="nearest")

ax3.set_xticks(range(24))
ax3.set_xticklabels([f"{h:02d}:00" for h in range(24)], rotation=45, ha="right", fontsize=FONTSIZE_TICK)
ax3.set_yticks(range(len(day_order)))
ax3.set_yticklabels(day_order, fontsize=FONTSIZE_TICK)
ax3.set_xlabel("Hour of day", fontsize=FONTSIZE_LABEL)

cbar = plt.colorbar(im, ax=ax3, shrink=0.8)
cbar.set_label("Average number of walks per day", fontsize=FONTSIZE_LABEL)
cbar.ax.tick_params(labelsize=FONTSIZE_TICK)

plt.tight_layout()
plt.show()
plt.close()

In [ ]:
n_occurrences_per_day

In [ ]:
day_counts

In [ ]:
day_counts_normalised

In [ ]:
time_counts_raw

In [ ]:
heatmap_data_normalised

### WALK DISTANCE ANALYSIS

#### PER DAY

In [ ]:
# ─── Occurrences par jour de semaine — TOUS modes ─────────────────────────────
days_count_per_weekday = (legs_GG_all
                          .groupby(["user_id_fors", "day_of_week", "day_of_week_num"])["legs_date"]
                          .nunique()
                          .reset_index()
                          .rename(columns={"legs_date": "n_weekday_occurrences"}))

# ─── Distance marchée par user par jour ───────────────────────────────────────
distance_walked = (legs_GG_walk_regular
                   .dropna(subset=["wgt_cant_trim_gps"])
                   .groupby(["user_id_fors", "day_of_week", "day_of_week_num", "age_fr_grouped"])["length_weighted"]
                   .sum()
                   .reset_index()
                   .rename(columns={"length_weighted": "total_distance_day"}))

# ─── Grille complète : tous les jours actifs + distance marchée (0 si pas marché) ──
distance_per_user_day = days_count_per_weekday.merge(
    distance_walked,
    on=["user_id_fors", "day_of_week", "day_of_week_num"],
    how="left"  # ← left sur days_count pour garder les jours sans marche
)

# Les jours sans marche ont total_distance_day = NaN → remplacer par 0
distance_per_user_day["total_distance_day"] = (
    distance_per_user_day["total_distance_day"].fillna(0)
)

# ─── age_fr_grouped manquante pour les jours sans marche → remplir depuis legs_GG_all ──
age_per_user = legs_GG_all.drop_duplicates('user_id_fors')[['user_id_fors', 'age_fr_grouped']]
distance_per_user_day = distance_per_user_day.merge(
    age_per_user, on='user_id_fors', how='left', suffixes=('', '_fill')
)
distance_per_user_day['age_fr_grouped'] = (
    distance_per_user_day['age_fr_grouped']
    .fillna(distance_per_user_day['age_fr_grouped_fill'])
)
distance_per_user_day = distance_per_user_day.drop(columns=['age_fr_grouped_fill'])

# ─── Distance moyenne par jour de semaine ─────────────────────────────────────
distance_per_user_day["mean_distance_single_day"] = (
    distance_per_user_day["total_distance_day"] /
    distance_per_user_day["n_weekday_occurrences"]
)

# ─── Stats tous utilisateurs ──────────────────────────────────────────────────
stats_by_day_user = (distance_per_user_day
                     .groupby("day_of_week")["mean_distance_single_day"]
                     .agg(median="median", mean="mean", count="count")
                     .reindex(day_order))

print("── Occurrences ─────────")
print(distance_per_user_day.groupby("day_of_week")["n_weekday_occurrences"].mean().reindex(day_order).round(1))
print("\n── Distance moyenne par jour de semaine — tous utilisateurs ──")
print(stats_by_day_user)

# ─── Stats par groupe d'âge ───────────────────────────────────────────────────
stats_by_day_age = (distance_per_user_day
                    .dropna(subset=["age_fr_grouped"])
                    .groupby(["day_of_week", "age_fr_grouped"])["mean_distance_single_day"]
                    .median()
                    .unstack("age_fr_grouped")
                    .reindex(day_order))

age_groups = stats_by_day_age.columns.tolist()

print("\n── Groupes d'âge ────────────────────────────────────")
print(stats_by_day_age.columns.tolist())
print(stats_by_day_age.round(0))

In [ ]:
days_count_per_weekday

In [ ]:
distance_per_user_day

In [ ]:
# ─── Nombre moyen de users actifs par occurrence du jour (tous modes) ─────────
mean_users_per_weekday = (legs_GG_all
    .groupby(["day_of_week", "legs_date"])["user_id_fors"]
    .nunique()
    .reset_index()
    .groupby("day_of_week")["user_id_fors"]
    .mean()
    .reindex(day_order)
    .round(0)
    .astype(int))

print("── Nombre moyen de users présents par occurrence du jour ──")
print(mean_users_per_weekday)

# ─── Stats tous utilisateurs ──────────────────────────────────────────────────
stats_by_day_user = (distance_per_user_day
                     .groupby("day_of_week")["mean_distance_single_day"]
                     .agg(median="median", mean="mean", count="count")
                     .reindex(day_order))

# ─── Ajouter le nombre moyen de users au dataframe de stats ──────────────────
stats_by_day_user["mean_users"] = mean_users_per_weekday

print("\n── Distance moyenne par jour de semaine — tous utilisateurs ──")
print(stats_by_day_user)

In [ ]:
# ─── Chart 1 : Bar chart ──────────────────────────────────────────────────────
fig1, ax1 = plt.subplots(figsize=(14, 6))
fig1.patch.set_alpha(0)
ax1.patch.set_alpha(0)

x     = np.arange(len(day_order))
width = 0.35

COLOR_DIST_MEDIAN =  "#aec7e8"
COLOR_DIST_MEAN   =  "#1f77b4"

bars_median = ax1.bar(x - width/2, stats_by_day_user["median"], width,
                       label="Median", color=COLOR_DIST_MEDIAN, alpha=0.85, edgecolor="white")
bars_mean   = ax1.bar(x + width/2, stats_by_day_user["mean"],   width,
                       label="Mean",   color=COLOR_DIST_MEAN,   alpha=0.85, edgecolor="white")

for bar in bars_median:
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
             f"{bar.get_height():.0f}m", ha="center", va="bottom", fontsize=8)
for bar in bars_mean:
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
             f"{bar.get_height():.0f}m", ha="center", va="bottom", fontsize=8)

ax1.set_xticks(x)
ax1.set_xticklabels(
    [f"{d}\n(n={mean_users_per_weekday.loc[d]:,.0f} user/days)" for d in day_order],
    fontsize=9)

ax1.axvspan(4.5, 6.5, alpha=0.08, color="orange")
ax1.text(5.5, ax1.get_ylim()[1] * 0.95, "Weekend", ha="center", fontsize=9, color="orange")
#ax1.set_title("Mean & Median cumulated walk distance per user per day", fontweight="bold")
ax1.set_ylabel("Distance (m)")
ax1.legend(fontsize=10)
ax1.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()
plt.close()


# ─── Chart 2 : Boxplot ────────────────────────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(14, 6))
fig2.patch.set_alpha(0)
ax2.patch.set_alpha(0)

data_by_day = [distance_per_user_day[distance_per_user_day["day_of_week"] == d]["mean_distance_single_day"].dropna().values
               for d in day_order]

bp = ax2.boxplot(
    data_by_day,
    labels=day_order,
    patch_artist=True,
    medianprops=dict(color="red", linewidth=2),
    flierprops=dict(marker="o", markersize=3, alpha=0.3),
    widths=0.5
)

for patch, color in zip(bp["boxes"], [COLOR_DIST_MEDIAN] * 7):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)

ax2.set_yscale("log")
ax2.axvspan(4.5, 6.5, alpha=0.08, color="orange")
ax2.text(5.5, ax2.get_ylim()[1] * 0.7, "Weekend", ha="center", fontsize=9, color="orange")
#ax2.set_title("Cumulated walk distance distribution per user per day", fontweight="bold")
ax2.set_ylabel("Distance (m) — log scale")
ax2.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()
plt.close()

In [ ]:
FONTSIZE_LABEL  = 12
FONTSIZE_TICK   = 12
FONTSIZE_LEGEND = 12
FONTSIZE_ANNOT  = 12

COLOR_DIST_MEAN   = "#1f77b4"
COLOR_DIST_MEDIAN = "#aec7e8"

fig, ax1 = plt.subplots(1, 1, figsize=(9, 6))
fig.patch.set_alpha(0)
ax1.patch.set_alpha(0)

x     = np.arange(len(day_order))
width = 0.35

bars_median = ax1.bar(x - width/2, stats_by_day_user["median"], width,
                      label="Median", color=COLOR_DIST_MEDIAN, alpha=0.85, edgecolor="white")
bars_mean   = ax1.bar(x + width/2, stats_by_day_user["mean"],   width,
                      label="Mean",   color=COLOR_DIST_MEAN,   alpha=0.85, edgecolor="white")

for bar in bars_median:
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
             f"{bar.get_height():.0f}m", ha="center", va="bottom", fontsize=FONTSIZE_ANNOT)
for bar in bars_mean:
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
             f"{bar.get_height():.0f}m", ha="center", va="bottom", fontsize=FONTSIZE_ANNOT)

ax1.set_xticks(x)
ax1.set_xticklabels(
    [f"{d}\n(n={mean_users_per_weekday.loc[d]:,.0f}\nuser/days)" for d in day_order],
    fontsize=FONTSIZE_TICK)

ax1.axvspan(4.5, 6.5, alpha=0.08, color="orange")
ax1.text(5.5, ax1.get_ylim()[1] * 0.96, "Weekend", ha="center", fontsize=FONTSIZE_ANNOT, color="orange")
ax1.set_ylabel("Distance (m)", fontsize=FONTSIZE_LABEL)
ax1.legend(fontsize=FONTSIZE_LEGEND)
ax1.grid(axis="y", alpha=0.3)
ax1.tick_params(axis="y", labelsize=FONTSIZE_TICK)

plt.tight_layout()
plt.show()
plt.close()

In [ ]:
stats_by_day_user

#### PER AGE GROUP

In [ ]:
PALETTE_AGE = ["#4C72B0", "#DD8452", "#55A868", "#C44E52",
               "#8172B2", "#937860", "#DA8BC3", "#8C8C8C"]

# ─── IQR par groupe d'âge par jour de semaine ────────────────────────────────
ci_per_age_day = (distance_per_user_day
                  .dropna(subset=["age_fr_grouped"])
                  .groupby(["day_of_week", "age_fr_grouped"])["mean_distance_single_day"]
                  .agg(
                      median="median",
                      q25=lambda x: x.quantile(0.25),
                      q75=lambda x: x.quantile(0.75)
                  )
                  .unstack("age_fr_grouped")
                  .reindex(day_order))

# ─── Nombre d'utilisateurs par groupe d'âge ──────────────────────────────────
n_per_age = (distance_per_user_day
             .dropna(subset=["age_fr_grouped"])
             .groupby("age_fr_grouped")["user_id_fors"]
             .nunique())

# ─── Graphiques ───────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(16, 14))
fig.suptitle("Distance de marche par utilisateur·rice par jour — Genève",
             fontsize=14, fontweight="bold")

ax1 = axes[0]
x   = np.arange(len(day_order))

for i, age in enumerate(age_groups):
    n      = n_per_age.get(age, 0)
    median = ci_per_age_day["median"][age].values
    q25    = ci_per_age_day["q25"][age].values
    q75    = ci_per_age_day["q75"][age].values
    color  = PALETTE_AGE[i % len(PALETTE_AGE)]

    style  = "--" if n < 50 else "-"
    marker = "x" if n < 50 else "o"

    ax1.plot(x, median,
             linestyle=style, marker=marker,
             linewidth=2, alpha=0.85, color=color,
             label=f"{age} (n={n}{'  ⚠️' if n < 50 else ''})")
    ax1.fill_between(x, q25, q75, alpha=0.12, color=color)

ax1.set_xticks(x)
ax1.set_xticklabels(day_order, fontsize=9)
ax1.axvspan(4.5, 6.5, alpha=0.08, color="orange")
ax1.text(5.5, ax1.get_ylim()[1] * 0.95, "Weekend",
         ha="center", fontsize=9, color="orange")
ax1.set_title("Distance médiane par utilisateur·rice par jour — par groupe d'âge (IQR Q25-Q75)",
              fontweight="bold")
ax1.set_ylabel("Distance médiane (m)")
ax1.legend(title="Groupe d'âge", bbox_to_anchor=(1.01, 1),
           loc="upper left", fontsize=9)
ax1.grid(alpha=0.3)

# ─── Boxplot ──────────────────────────────────────────────────────────────────
ax2 = axes[1]

data_by_age = [distance_per_user_day[
                   distance_per_user_day["age_fr_grouped"] == age
               ]["mean_distance_single_day"].dropna().values
               for age in age_groups]

bp = ax2.boxplot(
    data_by_age,
    labels=[f"{age}\n(n={n_per_age.get(age, 0)})" for age in age_groups],
    patch_artist=True,
    medianprops=dict(color="red", linewidth=2),
    flierprops=dict(marker="o", markersize=3, alpha=0.3),
    widths=0.5
)

for patch, color in zip(bp["boxes"], PALETTE_AGE):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)

ax2.set_yscale("log")
ax2.set_title("Distribution de la distance de marche par groupe d'âge (tous les jours)",
              fontweight="bold")
ax2.set_ylabel("Distance (m) — échelle log")
ax2.tick_params(axis="x", rotation=30)
ax2.grid(axis="y", alpha=0.3, which="both")

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(18, 7))
fig.suptitle("Cumulated walk Distance per User per Day by Age Group — Geneva",
             fontsize=14, fontweight="bold")

age_groups  = stats_by_day_age.columns.tolist()
n_ages      = len(age_groups)
x           = np.arange(len(day_order))
width       = 0.8 / (n_ages + 1)

# ─── Bars per age group with IQR error bars ───────────────────────────────────
for i, age in enumerate(age_groups):
    n      = n_per_age.get(age, 0)
    offset = (i - n_ages / 2) * width
    
    median = ci_per_age_day["median"][age].reindex(day_order).values
    q25    = ci_per_age_day["q25"][age].reindex(day_order).values
    q75    = ci_per_age_day["q75"][age].reindex(day_order).values

    # Asymmetric error : distance from median to Q25 (below) and Q75 (above)
    yerr_lower = median - q25
    yerr_upper = q75    - median

    ax.bar(x + offset, median,
           width=width * 0.9,
           color=PALETTE_AGE[i % len(PALETTE_AGE)],
           alpha=0.85, edgecolor="white",
           label=f"{age} (n={n}{'  ⚠️' if n < 50 else ''})",
           yerr=[yerr_lower, yerr_upper],
           error_kw=dict(ecolor="black", capsize=2, linewidth=0.8, alpha=0.6))

# ─── Overall median line ──────────────────────────────────────────────────────
ax.plot(x, stats_by_day_user["median"].reindex(day_order).values,
        "o--", color="red", linewidth=1, markersize=5,
        label="Overall median", zorder=5)

# ─── Overall mean line ────────────────────────────────────────────────────────
ax.plot(x, stats_by_day_user["mean"].reindex(day_order).values,
        "s--", color="orange", linewidth=1, markersize=5,
        label="Overall mean", zorder=5)

# ─── Formatting ───────────────────────────────────────────────────────────────
ax.set_xticks(x)
ax.set_xticklabels(day_order, fontsize=10)
ax.axvspan(4.5, 6.5, alpha=0.08, color="orange")
ax.text(5.5, ax.get_ylim()[1] * 0.95, "Weekend",
        ha="center", fontsize=9, color="orange")
ax.set_title("Median walk distance per age group per day (error bars = IQR Q25-Q75)",
             fontweight="bold")
ax.set_ylabel("Distance (m)")
ax.legend(title="Age group", bbox_to_anchor=(1.01, 1),
          loc="upper left", fontsize=9)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
#plt.savefig("walk_distance_age_combined.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ─── Tout est déjà calculé dans users_GG_walk_regular ────────────────────────
distance_per_user = users_GG_walk_regular[[
    'user_id_fors',
    'dist_walk',      # = total_distance
    'n_days_GG',      # = n_active_days (tous modes)
    'n_walk_days',    # = jours marchés uniquement
    'dist_walk_per_day',       # = dist_walk / n_days_GG
    'dist_walk_per_walk_day',  # = dist_walk / n_walk_days
]].copy().rename(columns={
    'dist_walk'  : 'total_distance',
    'n_days_GG'  : 'n_active_days',
})

# ─── Résumé ───────────────────────────────────────────────────────────────────
for var, label in [
    ("dist_walk_per_day",      "Distance marchée / jours actifs tous modes"),
    ("dist_walk_per_walk_day", "Distance marchée / jours marchés uniquement"),
]:
    print(f"\n── {label} ──────────────────────────────────────")
    print(f"Utilisateur·rices : {distance_per_user[var].notna().sum()}")
    print(f"Médiane (m)       : {distance_per_user[var].median():.1f}")
    print(f"Moyenne (m)       : {distance_per_user[var].mean():.1f}")
    print(f"Min     (m)       : {distance_per_user[var].min():.1f}")
    print(f"Max     (m)       : {distance_per_user[var].max():.1f}")

In [ ]:
distance_per_user.head(20)

In [ ]:
# ─── Utilisateur avec la distance max par jour de marche ─────────────────────
max_idx  = users_GG_walk_regular['dist_walk_per_walk_day'].idxmax()
max_user = users_GG_walk_regular.loc[max_idx]

print("── Utilisateur·rice avec distance max par jour de marche ──")
print(max_user[['user_id_fors', 'dist_walk_per_walk_day', 
                'dist_walk', 'n_walk_days', 'n_legs_walk']].to_string())

# ─── Détail des legs de cet·te utilisateur·rice ──────────────────────────────
print(f"\n── Legs pour l'utilisateur·rice {max_user['user_id_fors']} ──")
print(legs_GG_walk_regular[legs_GG_walk_regular["user_id_fors"] == max_user["user_id_fors"]]
      [["leg_id", "length_leg", "duration_min", "leading_stay_purpose", "legs_date"]]
      .sort_values("length_leg", ascending=False)
      .head(10)
      .to_string())

In [ ]:
from scipy.stats import gaussian_kde

# ─── plot_data depuis users_GG_walk_regular ───────────────────────────────────
plot_data = users_GG_walk_regular[[
    'user_id_fors',
    'dist_walk_per_day',         # distance marchée / tous les jours actifs GE
    'mean_walk_legs_per_day',
    'mean_walk_duration_per_day',
]].copy()

# ─── Graphique 3x2 ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 2, figsize=(14, 15))
fig.suptitle(
    f"Statistiques quotidiennes de marche par utilisateur·rice\n"
    f"(users_GG_walk_regular — n={len(plot_data)})",
    fontsize=14, fontweight="bold"
)

PALETTE = ["#4C72B0", "#DD8452", "#55A868"]

datasets = [
    ("dist_walk_per_day",         "Distance cumulée moyenne par jour actif dans GE", "Distance (m)"),
    ("mean_walk_legs_per_day",    "Nombre moyen de legs de marche par jour",          "Nombre de legs"),
    ("mean_walk_duration_per_day","Durée cumulée moyenne de marche par jour",         "Durée (min)"),
]

for i, (col, title, ylabel) in enumerate(datasets):
    ax_hist = axes[i, 0]
    ax_box  = axes[i, 1]
    color   = PALETTE[i]

    median = plot_data[col].median()
    mean   = plot_data[col].mean()

    if i in [0, 2]:
        data_clean = plot_data[col].dropna()
        kde = gaussian_kde(data_clean)
        x   = np.linspace(data_clean.min(), data_clean.max(), 500)
        ax_hist.plot(x, kde(x), color=color, linewidth=2)
        ax_hist.fill_between(x, kde(x), alpha=0.3, color=color)
        ax_hist.set_xscale("log")
        ax_hist.set_xlabel(ylabel + " — échelle log")
        ax_hist.set_ylabel("Densité")
    else:
        ax_hist.hist(plot_data[col].dropna(), bins=30, color=color, edgecolor="white", alpha=0.85)
        ax_hist.set_xlabel(ylabel)
        ax_hist.set_ylabel("Nombre d'utilisateur·rices")

    ax_hist.axvline(median, color="red",    linestyle="--", label=f"Médiane : {median:.1f}")
    ax_hist.axvline(mean,   color="orange", linestyle="--", label=f"Moyenne : {mean:.1f}")
    ax_hist.set_title(title, fontweight="bold")
    ax_hist.legend(fontsize=9)
    ax_hist.grid(axis="y", alpha=0.3)

    bp = ax_box.boxplot(plot_data[col].dropna().values,
                        vert=True, patch_artist=True, widths=0.5,
                        medianprops=dict(color="red", linewidth=2),
                        flierprops=dict(marker="o", markersize=3, alpha=0.4))
    bp["boxes"][0].set_facecolor(color)
    bp["boxes"][0].set_alpha(0.75)

    if i in [0, 2]:
        ax_box.set_yscale("log")
        ax_box.set_ylabel(ylabel + " — échelle log")
    else:
        ax_box.set_ylabel(ylabel)

    ax_box.axhline(mean, color="orange", linestyle="--", linewidth=1.2)
    ax_box.text(1.32, median, f"Médiane : {median:.1f}", va="center", color="red",    fontsize=9)
    ax_box.text(1.32, mean,   f"Moyenne : {mean:.1f}",   va="center", color="orange", fontsize=9)
    ax_box.set_title(f"{title} — distribution", fontweight="bold")
    ax_box.set_xticks([])
    ax_box.grid(axis="y", alpha=0.3, which="both")

plt.tight_layout()
plt.show()

In [ ]:
datasets = [
    ("dist_walk_per_day",         "Average cumulated walk distance per active day", "Distance (m)",   "m",   
     np.arange(0, 45000, 1000),   np.arange(0, 5001, 250)),   # bins global, bins zoom
    ("mean_walk_legs_per_day",    "Average number of walk legs per day",             "Number of legs", "",    
     np.arange(0, 30, 1),        None),                                              # pas de zoom
    ("mean_walk_duration_per_day","Average cumulated walk duration per active day",  "Duration (min)", "min", 
     np.arange(0, 500, 10),      np.arange(0, 121, 10)),     # bins global, bins zoom
]

In [ ]:
FONTSIZE_LABEL  = 12
FONTSIZE_TICK   = 12
FONTSIZE_LEGEND = 12
FONTSIZE_ANNOT  = 12

for i, (col, title, ylabel, unit, bins_global, bins_zoom) in enumerate(datasets):
    color  = PALETTE[i]
    median = plot_data[col].median()
    mean   = plot_data[col].mean()

    # ─── Histogram ───────────────────────────────────────────────────────────
    fig, ax_hist = plt.subplots(figsize=(7, 5))
    fig.patch.set_alpha(0)
    ax_hist.patch.set_alpha(0)

    data_clean = plot_data[col].dropna()
    weights    = np.ones(len(data_clean)) / len(data_clean) * 100

    ax_hist.hist(data_clean, bins=bins_global, weights=weights, color=color, edgecolor="white", alpha=0.85)
    ax_hist.set_ylabel("Percentage of users (%)", fontsize=FONTSIZE_LABEL)
    ax_hist.set_xlabel(ylabel, fontsize=FONTSIZE_LABEL)
    ax_hist.tick_params(axis='both', labelsize=FONTSIZE_TICK)
    ax_hist.axvline(median, color="red",    linestyle="--", label=f"Median: {median:.1f} {unit}")
    ax_hist.axvline(mean,   color="orange", linestyle="--", label=f"Mean: {mean:.1f} {unit}")
    ax_hist.legend(fontsize=FONTSIZE_LEGEND)
    ax_hist.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()
    plt.close()

    # ─── Histogram zoomé ─────────────────────────────────────────────────────
    if i in [0, 2]:
        ZOOM_MAX     = 5000 if i == 0 else 120
        data_zoom    = plot_data[col].dropna()
        n_excluded   = (data_zoom > ZOOM_MAX).sum()
        pct_excluded = n_excluded / len(data_zoom) * 100
        zoom_label   = f"0–{ZOOM_MAX:,}m" if i == 0 else f"0–{ZOOM_MAX} min"

        print(f"   {n_excluded} users excluded ({pct_excluded:.1f}% of sample)")

        data_zoom_filtered = data_zoom[data_zoom <= ZOOM_MAX]
        weights_zoom       = np.ones(len(data_zoom_filtered)) / len(data_zoom) * 100

        fig, ax_zoom = plt.subplots(figsize=(7, 5))
        fig.patch.set_alpha(0)
        ax_zoom.patch.set_alpha(0)

        ax_zoom.hist(data_zoom_filtered, bins=bins_zoom, weights=weights_zoom,
                     color=color, edgecolor="white", alpha=0.85)
        ax_zoom.set_ylabel("Percentage of users (%)", fontsize=FONTSIZE_LABEL)
        ax_zoom.set_xlabel(ylabel, fontsize=FONTSIZE_LABEL)
        ax_zoom.tick_params(axis='both', labelsize=FONTSIZE_TICK)
        ax_zoom.axvline(median, color="red",    linestyle="--", label=f"Median: {median:.1f} {unit}")
        ax_zoom.axvline(mean,   color="orange", linestyle="--", label=f"Mean: {mean:.1f} {unit}")
        ax_zoom.legend(fontsize=FONTSIZE_LEGEND)
        ax_zoom.grid(axis="y", alpha=0.3)
        plt.tight_layout()
        plt.show()
        plt.close()

    # ─── Boxplot ─────────────────────────────────────────────────────────────
    fig, ax_box = plt.subplots(figsize=(5, 5))
    fig.patch.set_alpha(0)
    ax_box.patch.set_alpha(0)

    bp = ax_box.boxplot(plot_data[col].dropna().values,
                        vert=True, patch_artist=True, widths=0.5,
                        medianprops=dict(color="red", linewidth=2),
                        flierprops=dict(marker="o", markersize=3, alpha=0.4))
    bp["boxes"][0].set_facecolor(color)
    bp["boxes"][0].set_alpha(0.75)

    if i in [0, 2]:
        ax_box.set_yscale("log")
        ax_box.set_ylabel(ylabel + " — log scale", fontsize=FONTSIZE_LABEL)
    else:
        ax_box.set_ylabel(ylabel, fontsize=FONTSIZE_LABEL)

    ax_box.tick_params(axis='y', labelsize=FONTSIZE_TICK)
    ax_box.axhline(mean, color="orange", linestyle="--", linewidth=1.2)
    ax_box.text(1.32, median, f"Median: {median:.1f} {unit}", va="top",    color="red",    fontsize=FONTSIZE_ANNOT)
    ax_box.text(1.32, mean,   f"Mean: {mean:.1f} {unit}",     va="bottom", color="orange", fontsize=FONTSIZE_ANNOT)
    ax_box.set_xticks([])
    ax_box.grid(axis="y", alpha=0.3, which="both")
    plt.tight_layout()
    plt.show()
    plt.close()

In [ ]:
# ─── Seuil outliers : utilisateurs au-dessus du percentile 95 ────────────────
p95 = users_GG_walk_regular['dist_walk_per_day'].quantile(0.95)
p99 = users_GG_walk_regular['dist_walk_per_day'].quantile(0.99)

print(f"P95 : {p95:.0f}m | P99 : {p99:.0f}m")

outliers = (users_GG_walk_regular[users_GG_walk_regular['dist_walk_per_day'] > p95]
            [['user_id_fors', 'dist_walk_per_day', 'dist_walk', 
              'n_walk_days', 'n_legs_walk']]
            .sort_values('dist_walk_per_day', ascending=False))

print(f"\n── {len(outliers)} utilisateur·rices au-dessus du P95 ({p95:.0f}m/jour) ──")
print(outliers.to_string())

# ─── Pour chaque outlier, regarder ses legs les plus longs ───────────────────
for user_id in outliers['user_id_fors'].head(10):
    user_legs = (legs_GG_walk_regular[legs_GG_walk_regular["user_id_fors"] == user_id]
                 [["leg_id", "length_leg", "duration_min", "leading_stay_purpose", "legs_date"]]
                 .sort_values("length_leg", ascending=False)
                 .head(5))
    dist = outliers.loc[outliers['user_id_fors'] == user_id, 'dist_walk_per_day'].values[0]
    print(f"\n── User {user_id} — {dist:.0f}m/jour ──")
    print(user_legs.to_string())

In [ ]:
# ─── Utilisateur avec la distance moyenne par jour maximale ──────────────────
max_user = users_GG_walk_regular.loc[
    users_GG_walk_regular['dist_walk_per_day'].idxmax()
]

print(f"── Utilisateur avec dist_walk_per_day maximale ──")
print(f"  user_id          : {max_user['user_id_fors']}")
print(f"  dist_walk_per_day: {max_user['dist_walk_per_day']:.0f}m")
print(f"  dist_walk (total): {max_user['dist_walk']:.0f}m")
print(f"  n_days_GG        : {max_user['n_days_GG']:.0f} jours")
print(f"  n_walk_days      : {max_user['n_walk_days']:.0f} jours")
print(f"  n_legs_walk      : {max_user['n_legs_walk']:.0f} legs")

# ─── Ses legs détaillés ───────────────────────────────────────────────────────
user_legs = (legs_GG_walk_regular[
                legs_GG_walk_regular["user_id_fors"] == max_user['user_id_fors']]
             [["leg_id", "length_leg", "duration_min", 
               "speed_kmh", "leading_stay_purpose", "legs_date"]]
             .sort_values("length_leg", ascending=False))

print(f"\n── Tous les legs de cet utilisateur ({len(user_legs)} legs) ──")
print(user_legs.to_string())

# ─── Stats par jour pour cet utilisateur ─────────────────────────────────────
daily_stats = (legs_GG_walk_regular[
                legs_GG_walk_regular["user_id_fors"] == max_user['user_id_fors']]
               .groupby("legs_date")
               .agg(
                   n_legs      = ("leg_id",       "count"),
                   dist_total  = ("length_leg",   "sum"),
                   dur_total   = ("duration_min", "sum"),
               )
               .reset_index())

print(f"\n── Stats par jour ──")
print(daily_stats.to_string())

In [ ]:
# ─── Stats par jour pour cet utilisateur ─────────────────────────────────────
max_user_id = users_GG_walk_regular.loc[
    users_GG_walk_regular['dist_walk_per_day'].idxmax(), 'user_id_fors'
]

daily_stats = (legs_GG_walk_regular[
                legs_GG_walk_regular["user_id_fors"] == max_user_id]
               .groupby("legs_date")
               .agg(
                   n_legs     = ("leg_id",       "count"),
                   dist_total = ("length_leg",   "sum"),
                   dur_total  = ("duration_min", "sum"),
                   speed_max  = ("speed_kmh",    "max"),
                   speed_med  = ("speed_kmh",    "median"),
               )
               .reset_index())

print(daily_stats.to_string())

In [ ]:
print(legs_GG_walk_regular[
    legs_GG_walk_regular["user_id_fors"] == max_user_id
][["leg_id", "length_leg", "length_weighted", "wgt_cant_trim_gps"]].head(10).to_string())

In [ ]:
# ─── Caractéristiques socio-démographiques du user max ───────────────────────
print(legs_GG_walk_regular[
    legs_GG_walk_regular["user_id_fors"] == max_user_id
][["user_id_fors", "gdr", "age_fr", "prof", "wgt_cant_trim_gps"]].drop_duplicates().to_string())

In [ ]:
# ─── Focus sur les grands outliers (> P99) ───────────────────────────────────
p99 = users_GG_walk_regular['dist_walk_per_day'].quantile(0.99)

big_outliers = (users_GG_walk_regular[users_GG_walk_regular['dist_walk_per_day'] > p99]
                ['user_id_fors'].tolist())

print(f"P99 : {p99:.0f}m — {len(big_outliers)} utilisateurs concernés")

# ─── Pour chaque grand outlier, statistiques sur ses legs ────────────────────
for user_id in big_outliers:
    user_legs = legs_GG_walk_regular[legs_GG_walk_regular["user_id_fors"] == user_id].copy()
    
    print(f"\n{'='*60}")
    print(f"User {user_id} — {len(user_legs)} legs au total")
    print(f"  Distance/jour       : {users_GG_walk_regular.loc[users_GG_walk_regular['user_id_fors']==user_id, 'dist_walk_per_day'].values[0]:.0f}m")
    print(f"  Longueur max leg    : {user_legs['length_leg'].max():.0f}m")
    print(f"  Longueur médiane    : {user_legs['length_leg'].median():.0f}m")
    print(f"  Durée max leg       : {user_legs['duration_min'].max():.1f}min")
    print(f"  Vitesse max (km/h)  : {(user_legs['length_leg'] / (user_legs['duration_min']/60) / 1000).max():.1f}")
    print(f"  Vitesse médiane     : {(user_legs['length_leg'] / (user_legs['duration_min']/60) / 1000).median():.1f}")
    print(f"\n  Top 5 legs les plus longs :")
    print(user_legs[["leg_id", "length_leg", "duration_min", "legs_date"]]
          .sort_values("length_leg", ascending=False)
          .head(5)
          .to_string(index=False))

In [ ]:
WALK_SPEED_MAX = 30  # km/h 

# ─── Ajouter la vitesse à legs_GG_walk ───────────────────────────────────────
legs_GG_walk["speed_kmh"] = (
    legs_GG_walk["length_leg"] / (legs_GG_walk["duration_min"] / 60) / 1000
)

# ─── Identifier les legs aberrants ───────────────────────────────────────────
legs_aberrants = legs_GG_walk[legs_GG_walk["speed_kmh"] > WALK_SPEED_MAX].copy()

n_total = len(legs_GG_walk)
n_aberrants = len(legs_aberrants)

print(f"Legs aberrants (> {WALK_SPEED_MAX} km/h) : {n_aberrants} ({n_aberrants/n_total*100:.2f}% des legs de marche)")
print(f"Utilisateurs concernés : {legs_aberrants['user_id_fors'].nunique()} ({legs_aberrants['user_id_fors'].nunique()/legs_GG_walk['user_id_fors'].nunique()*100:.1f}% des utilisateurs)")
print(f"\nDistribution des vitesses aberrantes :")
print(legs_aberrants["speed_kmh"].describe())
print(f"\nTop 10 legs les plus rapides :")
print(legs_aberrants[["user_id_fors", "leg_id", "length_leg", "duration_min", "speed_kmh", "legs_date"]]
      .sort_values("speed_kmh", ascending=False)
      .head(10)
      .to_string(index=False))

In [ ]:
# ─── Diagnostic des legs aberrants : temps ou distance ? ─────────────────────
print("Statistiques sur les legs aberrants :")
print(legs_aberrants[["length_leg", "duration_min", "speed_kmh"]].describe())

# ─── Visualisation : est-ce que la distance est normale ? ────────────────────
print("\nComparaison avec les legs normaux :")
legs_normaux = legs_GG_walk[legs_GG_walk["speed_kmh"] <= 30].copy()

print(f"  Longueur médiane — normaux    : {legs_normaux['length_leg'].median():.0f}m")
print(f"  Longueur médiane — aberrants  : {legs_aberrants['length_leg'].median():.0f}m")
print(f"  Durée médiane    — normaux    : {legs_normaux['duration_min'].median():.1f}min")
print(f"  Durée médiane    — aberrants  : {legs_aberrants['duration_min'].median():.1f}min")

# ─── Cas extrêmes : durée quasi nulle ? ──────────────────────────────────────
print(f"\nLegs aberrants avec durée < 1 min : {(legs_aberrants['duration_min'] < 1).sum()}")
print(f"Legs aberrants avec durée < 0.5 min : {(legs_aberrants['duration_min'] < 0.5).sum()}")

In [ ]:
# ─── Vérification des flags extreme sur les legs aberrants ───────────────────
extreme_cols = [col for col in legs_GG_walk.columns if col.startswith("extreme")]
print("Colonnes extreme disponibles :", extreme_cols)

# ─── Flags sur les legs aberrants ────────────────────────────────────────────
legs_aberrants_flags = legs_aberrants[["user_id_fors", "leg_id", "speed_kmh"] + extreme_cols].copy()

print(f"\nRépartition des flags extreme sur les {len(legs_aberrants)} legs aberrants :")
for col in extreme_cols:
    n_flagged = legs_aberrants_flags[col].sum()
    print(f"  {col} = 1 : {n_flagged} legs ({n_flagged/len(legs_aberrants)*100:.1f}%)")

print(f"\nLegs aberrants avec aucun flag extreme :")
no_flag = legs_aberrants_flags[(legs_aberrants_flags[extreme_cols] == 0).all(axis=1)]
print(f"  {len(no_flag)} legs ({len(no_flag)/len(legs_aberrants)*100:.1f}%) — non détectés par les filtres PL")

In [ ]:
# ─── Leg le plus long en durée pour le user max ──────────────────────────────
user_legs_detail = (legs_GG_walk_regular[
    legs_GG_walk_regular["user_id_fors"] == max_user_id]
    [["leg_id", "length_leg", "length_weighted", "duration_min", 
      "speed_kmh", "legs_date", "leading_stay_purpose"]]
    .sort_values("duration_min", ascending=False))

print(f"── Top 10 legs les plus longs en durée ──")
print(user_legs_detail.head(10).to_string())

print(f"\n── Top 10 legs les plus longs en distance ──")
print(user_legs_detail.sort_values("length_leg", ascending=False).head(10).to_string())

print(f"\n── Stats globales sur ses legs ──")
print(f"  Durée max        : {user_legs_detail['duration_min'].max():.1f} min")
print(f"  Durée médiane    : {user_legs_detail['duration_min'].median():.1f} min")
print(f"  Distance max     : {user_legs_detail['length_leg'].max():.0f} m")
print(f"  Distance médiane : {user_legs_detail['length_leg'].median():.0f} m")
print(f"  Speed max        : {user_legs_detail['speed_kmh'].max():.1f} km/h")
print(f"  Speed médiane    : {user_legs_detail['speed_kmh'].median():.1f} km/h")

In [ ]:
plot_data

In [ ]:
# Comparer pour un utilisateur donné
user_test = legs_GG_all['user_id_fors'].iloc[897] #test random

print("Tous les lundis actifs (legs_GG_all) :")
print(legs_GG_all[
    (legs_GG_all['user_id_fors'] == user_test) &
    (legs_GG_all['day_of_week'] == 'Monday')
]['legs_date'].nunique())

print("\nLundis marchés uniquement (legs_GG_walk_regular) :")
print(legs_GG_walk_regular[
    (legs_GG_walk_regular['user_id_fors'] == user_test) &
    (legs_GG_walk_regular['day_of_week'] == 'Monday')
]['legs_date'].nunique())

print("\nn_weekday_occurrences pour ce user un lundi :")
print(days_count_per_weekday[
    (days_count_per_weekday['user_id_fors'] == user_test) &
    (days_count_per_weekday['day_of_week'] == 'Monday')
]['n_weekday_occurrences'].values)

### WALK DURATION

In [ ]:
# ─── Durée marchée par user par jour ──────────────────────────────────────────
_duration_walked = (legs_GG_walk_regular
                    .dropna(subset=["wgt_cant_trim_gps"])
                    .groupby(["user_id_fors", "day_of_week", "day_of_week_num"])["duration_min_weighted"]
                    .sum()
                    .reset_index()
                    .rename(columns={"duration_min_weighted": "total_duration_day"}))

# ─── Grille complète : tous jours actifs GE + durée (0 si pas marché) ─────────
duration_per_user_day = days_count_per_weekday.merge(
    _duration_walked,
    on=["user_id_fors", "day_of_week", "day_of_week_num"],
    how="left"   # ← left sur days_count_per_weekday cette fois
)
del _duration_walked

duration_per_user_day["total_duration_day"] = (
    duration_per_user_day["total_duration_day"].fillna(0)
)

duration_per_user_day["mean_duration_single_day"] = (
    duration_per_user_day["total_duration_day"] /
    duration_per_user_day["n_weekday_occurrences"]
)

stats_by_day_user_duration = (duration_per_user_day
                               .groupby("day_of_week")["mean_duration_single_day"]
                               .agg(median="median", mean="mean", count="count")
                               .reindex(day_order))

print("\n── Durée pondérée moyenne par jour de semaine par utilisateur·rice (min) ──")
print(stats_by_day_user_duration)

In [ ]:
COLOR_DUR_MEAN   = "#2ca02c"  # vert foncé matplotlib
COLOR_DUR_MEDIAN = "#98df8a"  # vert clair

# ─── Chart 1 : Bar chart ──────────────────────────────────────────────────────
print("\n── Cumulated walk duration per user per day — bar chart ─────────────")
fig1, ax1 = plt.subplots(figsize=(14, 6))
fig1.patch.set_alpha(0)
ax1.patch.set_alpha(0)

x     = np.arange(len(day_order))
width = 0.35

bars_median = ax1.bar(x - width/2, stats_by_day_user_duration["median"], width,
                       label="Median", color=COLOR_DUR_MEDIAN, alpha=0.85, edgecolor="white")
bars_mean   = ax1.bar(x + width/2, stats_by_day_user_duration["mean"],   width,
                       label="Mean",   color=COLOR_DUR_MEAN,   alpha=0.85, edgecolor="white")

for bar in bars_median:
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f"{bar.get_height():.1f}min", ha="center", va="bottom", fontsize=8)
for bar in bars_mean:
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f"{bar.get_height():.1f}min", ha="center", va="bottom", fontsize=8)

ax1.set_xticks(x)
ax1.set_xticklabels(
    [f"{d}\n(n={mean_users_per_weekday.loc[d]:,.0f} user/days)" for d in day_order],
    fontsize=9)

ax1.axvspan(4.5, 6.5, alpha=0.08, color="orange")
ax1.text(5.5, ax1.get_ylim()[1] * 0.95, "Weekend", ha="center", fontsize=9, color="orange")
#ax1.set_title("Mean & Median cumulated walk duration per user per day", fontweight="bold")
ax1.set_ylabel("Duration (min)")
ax1.legend(fontsize=10)
ax1.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()
plt.close()


# ─── Chart 2 : Boxplot ────────────────────────────────────────────────────────
print("\n── Cumulated walk duration per user per day — boxplot ───────────────")
fig2, ax2 = plt.subplots(figsize=(14, 6))
fig2.patch.set_alpha(0)
ax2.patch.set_alpha(0)

data_by_day = [duration_per_user_day[duration_per_user_day["day_of_week"] == d]["mean_duration_single_day"].dropna().values
               for d in day_order]

bp = ax2.boxplot(
    data_by_day,
    labels=day_order,
    patch_artist=True,
    medianprops=dict(color="red", linewidth=2),
    flierprops=dict(marker="o", markersize=3, alpha=0.3),
    widths=0.5
)

for patch in bp["boxes"]:
    patch.set_facecolor(COLOR_DUR_MEAN)
    patch.set_alpha(0.75)

ax2.set_yscale("log")
ax2.axvspan(4.5, 6.5, alpha=0.08, color="orange")
ax2.text(5.5, ax2.get_ylim()[1] * 0.7, "Weekend", ha="center", fontsize=9, color="orange")
#ax2.set_title("Cumulated walk duration distribution per user per day", fontweight="bold")
ax2.set_ylabel("Duration (min) — log scale")
ax2.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()
plt.close()

In [ ]:
FONTSIZE_LABEL  = 12
FONTSIZE_TICK   = 12
FONTSIZE_LEGEND = 12
FONTSIZE_ANNOT  = 12

fig, ax1 = plt.subplots(1, 1, figsize=(9, 6))
fig.patch.set_alpha(0)
ax1.patch.set_alpha(0)

x     = np.arange(len(day_order))
width = 0.35

bars_median = ax1.bar(x - width/2, stats_by_day_user_duration["median"], width,
                      label="Median", color=COLOR_DUR_MEDIAN, alpha=0.85, edgecolor="white")
bars_mean   = ax1.bar(x + width/2, stats_by_day_user_duration["mean"],   width,
                      label="Mean",   color=COLOR_DUR_MEAN,   alpha=0.85, edgecolor="white")

for bar in bars_median:
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f"{bar.get_height():.1f}min", ha="center", va="bottom", fontsize=FONTSIZE_ANNOT)
for bar in bars_mean:
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f"{bar.get_height():.1f}min", ha="center", va="bottom", fontsize=FONTSIZE_ANNOT)

ax1.set_xticks(x)
ax1.set_xticklabels(
    [f"{d}\n(n={mean_users_per_weekday.loc[d]:,.0f}\nuser/days)" for d in day_order],
    fontsize=FONTSIZE_TICK)

ax1.axvspan(4.5, 6.5, alpha=0.08, color="orange")
ax1.text(5.5, ax1.get_ylim()[1] * 0.95, "Weekend", ha="center", fontsize=FONTSIZE_ANNOT, color="orange")
ax1.set_ylabel("Duration (min)", fontsize=FONTSIZE_LABEL)
ax1.legend(fontsize=FONTSIZE_LEGEND)
ax1.grid(axis="y", alpha=0.3)
ax1.tick_params(axis="both", labelsize=FONTSIZE_TICK)

plt.tight_layout()
plt.show()
plt.close()

In [ ]:
# ─── Vérification cohérence des métriques ─────────────────────────────────────
_u = users_GG_walk_regular

print("── Depuis users_GG_walk_regular (valeur par personne) ──")
print(f"dist_walk_per_day médiane          : {_u['dist_walk_per_day'].median():.0f} m")
print(f"mean_walk_duration_per_day médiane : {_u['mean_walk_duration_per_day'].median():.1f} min")
print(f"mean_walk_legs_per_day médiane     : {_u['mean_walk_legs_per_day'].median():.1f} legs")

print("\n── Depuis stats_by_day_user (moyenne des jours de semaine) ──")
print(f"distance médiane moyenne           : {stats_by_day_user['median'].mean():.0f} m")
print(f"durée médiane moyenne              : {stats_by_day_user_duration['median'].mean():.1f} min")

del _u

In [ ]:
_u = users_GG_walk_regular

print("── Distribution dist_walk_per_day ──────────────────")
print(_u['dist_walk_per_day'].describe())

print("\n── Combien de personnes avec dist_walk_per_day < 100m ──")
print(f"{(_u['dist_walk_per_day'] < 100).sum()} / {len(_u)}")

print("\n── Combien de personnes avec n_walk_days == 0 ──────")
print(f"{(_u['n_walk_days'] == 0).sum()} / {len(_u)}")

del _u

In [ ]:
_u = users_GG_walk_regular

print("── Percentiles dist_walk_per_day ────────────────────")
for p in [5, 10, 25, 50, 75, 90, 95]:
    val = _u['dist_walk_per_day'].quantile(p/100)
    print(f"  P{p:>2} : {val:>8.0f} m/jour")

print("\n── Distribution par tranche ─────────────────────────")
tranches = [0, 100, 500, 1000, 2000, 5000, 999999]
labels   = ['< 100m', '100-500m', '500m-1km', '1-2km', '2-5km', '> 5km']
for i, (low, high, label) in enumerate(zip(tranches[:-1], tranches[1:], labels)):
    n   = ((_u['dist_walk_per_day'] >= low) & (_u['dist_walk_per_day'] < high)).sum()
    pct = n / len(_u) * 100
    print(f"  {label:<12} : {n:>4} ({pct:.1f}%)")

print(f"\n── Rapport n_walk_days / n_days_GE ──────────────────")
_u['walk_day_ratio'] = _u['n_walk_days'] / _u['n_days_GG']
print(_u['walk_day_ratio'].describe())

del _u

### PURPOSE ANALYSIS

In [ ]:
# ─── Walking purpose analysis ─────────────────────────────────────────────────

# 1. Quick overview
print("── Leading Stay Purpose ─────────────────────────────")
print(f"Total walks        : {len(legs_GG_walk_regular)}")
print(f"NaN purpose        : {legs_GG_walk_regular['leading_stay_purpose'].isna().sum()}")
print(f"Unique purposes    : {legs_GG_walk_regular['leading_stay_purpose'].nunique()}")
print()
print(legs_GG_walk_regular["leading_stay_purpose"].value_counts())

In [ ]:
# ─── Preparation ──────────────────────────────────────────────────────────────
legs_GG_walk_regular["purpose_group"] = legs_GG_walk_regular["leading_stay_purpose"].map(PURPOSE_GROUP_MAP)

n_total = len(legs_GG_walk_regular)
n_nan   = legs_GG_walk_regular["leading_stay_purpose"].isna().sum()
n_valid = legs_GG_walk_regular["leading_stay_purpose"].notna().sum()

print(f"── Vérification ─────────────────────────────────────")
print(f"Total legs         : {n_total:,}")
print(f"Valid (non-NaN)    : {n_valid:,} ({n_valid/n_total*100:.1f}%)")
print(f"NaN                : {n_nan:,}  ({n_nan/n_total*100:.1f}%)")
print(f"Contrôle somme     : {n_valid + n_nan:,} == {n_total:,} → {n_valid + n_nan == n_total}")

purpose_counts = legs_GG_walk_regular["leading_stay_purpose"].value_counts(dropna=True)
group_counts   = legs_GG_walk_regular["purpose_group"].value_counts()

print(f"\n── Répartition par groupe ───────────────────────────")
for g in PURPOSE_GROUP_ORDER:
    n = group_counts.get(g, 0)
    print(f"  {g:<20} : {n:,} ({n/n_valid*100:.1f}%)")
print(f"  {'Total valides':<20} : {group_counts.sum():,}")

# ─── Legend patches avec % par groupe ────────────────────────────────────────
legend_patches = [
    mpatches.Patch(
        color=PURPOSE_GROUP_COLORS[g],
        label=f"{g} ({group_counts.get(g, 0) / n_valid * 100:.1f}%)"
    )
    for g in PURPOSE_GROUP_ORDER if g in group_counts.index
]

# ─── Chart 1 : Horizontal bar chart — all purposes ────────────────────────────
print("\n── Walking Purpose — all purposes — horizontal bar chart ────────────")
fig1, ax1 = plt.subplots(figsize=(8, 7))
fig1.patch.set_alpha(0)
ax1.patch.set_alpha(0)

bar_colors = [PURPOSE_GROUP_COLORS.get(PURPOSE_GROUP_MAP.get(cat, "Other"), "#8C8C8C")
              for cat in purpose_counts.index]

bars = ax1.barh(
    purpose_counts.index[::-1],
    purpose_counts.values[::-1],
    color=bar_colors[::-1],
    edgecolor="white", linewidth=0.8, alpha=0.88
)

for bar, val in zip(bars, purpose_counts.values[::-1]):
    pct = val / n_valid * 100
    ax1.text(bar.get_width() + 50, bar.get_y() + bar.get_height()/2,
             f"{val:,}  ({pct:.1f}%)", va="center", fontsize=10)

#ax1.set_title(f"Walk legs by purpose\n({n_valid:,} walks / {n_nan:,} NaN)",
              #fontweight="bold")  # commenter pour export rapport
ax1.set_xlabel("Number of walk legs", fontsize=11)
ax1.tick_params(axis="both", labelsize=10)
ax1.grid(axis="x", alpha=0.3)
ax1.set_xlim(0, purpose_counts.max() * 1.35)
ax1.legend(handles=legend_patches, fontsize=10, loc="lower right", 
           frameon=True, title="Constraint group", title_fontsize=10)

plt.tight_layout()
plt.show()
plt.close()


# ─── Chart 2 : Pie chart — grouped categories ─────────────────────────────────
print("\n── Walking Purpose — constraint groups — pie chart ──────────────────")
fig2, ax2 = plt.subplots(figsize=(7, 7))
fig2.patch.set_alpha(0)
ax2.patch.set_alpha(0)

group_counts_ordered = group_counts.reindex(
    [g for g in PURPOSE_GROUP_ORDER if g in group_counts.index]
)
group_colors_list = [PURPOSE_GROUP_COLORS[g] for g in group_counts_ordered.index]

wedges, texts, autotexts = ax2.pie(
    group_counts_ordered.values,
    labels=group_counts_ordered.index,
    autopct=lambda p: f"{p:.1f}%",
    colors=group_colors_list,
    startangle=90,
    wedgeprops=dict(edgecolor="white", linewidth=2),
    pctdistance=0.75,
    textprops={"fontsize": 11}
)
for at in autotexts:
    at.set_fontsize(11)
    at.set_fontweight("bold")

#ax2.set_title("Walk legs by constraint group\n(excl. NaN)",
              #fontweight="bold")  # commenter pour export rapport

plt.tight_layout()
plt.show()
plt.close()


# ─── Chart 3 : Stacked bar — constraint group by gender ───────────────────────
print("\n── Walking Purpose — constraint group by gender — stacked bar ───────")
fig3, ax3 = plt.subplots(figsize=(7, 6))
fig3.patch.set_alpha(0)
ax3.patch.set_alpha(0)

cross = (legs_GG_walk_regular[legs_GG_walk_regular["purpose_group"].notna()]
         .groupby(["gdr", "purpose_group"])
         .size()
         .unstack(fill_value=0)
         .apply(lambda x: x / x.sum() * 100, axis=1))

cols_available = [g for g in PURPOSE_GROUP_ORDER if g in cross.columns]
cross[cols_available].plot(
    kind="bar", stacked=True, ax=ax3,
    color=[PURPOSE_GROUP_COLORS[g] for g in cols_available],
    edgecolor="white", linewidth=0.5, alpha=0.88
)

for bar_idx, (gender, row) in enumerate(cross[cols_available].iterrows()):
    cumulative = 0
    for group in cols_available:
        val = row[group]
        if val > 3:
            ax3.text(
                bar_idx,
                cumulative + val / 2,
                f"{val:.1f}%",
                ha="center", va="center",
                fontsize=10, fontweight="bold", color="white"
            )
        cumulative += val

#ax3.set_title("Constraint group distribution by gender\n(% normalized)",
              #fontweight="bold")  # commenter pour export rapport
ax3.set_xlabel("Gender", fontsize=11)
ax3.set_ylabel("% of walk legs", fontsize=11)
ax3.set_xticklabels(ax3.get_xticklabels(), rotation=0, fontsize=10)
ax3.tick_params(axis="y", labelsize=10)
ax3.legend(title="Constraint group", bbox_to_anchor=(1.01, 1),
           loc="upper left", fontsize=10)
ax3.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()
plt.close()

In [ ]:
purpose_hour = (legs_GG_walk_regular
                .groupby(["leading_stay_purpose", "hour_end"]) 
                .size()
                .unstack(fill_value=0))

purpose_hour_pct = purpose_hour.div(purpose_hour.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(16, 8))
im = ax.imshow(purpose_hour_pct.values, aspect="auto", cmap="YlOrRd")

ax.set_xticks(range(24))
ax.set_xticklabels([f"{h:02d}:00" for h in range(24)], rotation=45, ha="right", fontsize=16)
ax.set_yticks(range(len(purpose_hour_pct.index)))
ax.set_yticklabels(purpose_hour_pct.index, fontsize=16)
#ax.set_title("Walk purpose distribution by arrival hour (%)", fontweight="bold", fontsize=16)
ax.set_xlabel("Arrival hour", fontsize=16)
cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label("% of leg", fontsize=16)
cbar.ax.tick_params(labelsize=14)  # ← taille des graduations
plt.tight_layout()
plt.show()

In [ ]:
purpose_hour = (legs_GG_walk_regular
                .groupby(["leading_stay_purpose", "hour_end"]) 
                .size()
                .unstack(fill_value=0))

purpose_hour_pct = purpose_hour.div(purpose_hour.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(14, 5))
im = ax.imshow(purpose_hour_pct.values, aspect="auto", cmap="YlOrRd")

ax.set_xticks(range(24))
ax.set_xticklabels([f"{h:02d}:00" for h in range(24)], rotation=45, ha="right", fontsize=10)
ax.set_yticks(range(len(purpose_hour_pct.index)))
ax.set_yticklabels(purpose_hour_pct.index, fontsize=10)
#ax.set_title("Walk purpose distribution by arrival hour (%)", fontweight="bold", fontsize=16)
ax.set_xlabel("Arrival hour", fontsize=10)
cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label("% of leg", fontsize=10)
cbar.ax.tick_params(labelsize=10)  # ← taille des graduations
plt.tight_layout()
plt.show()

In [ ]:
purpose_hour

In [ ]:
purpose_day = (legs_GG_walk_regular
               .groupby(["day_of_week", "leading_stay_purpose"])
               .size()
               .unstack(fill_value=0)
               .reindex(day_order)
               .apply(lambda x: x / x.sum() * 100, axis=1))

# toutes les activités triées par fréquence décroissante
all_purposes = purpose_counts.index  # déjà trié par fréquence

# ─── Palette avec autant de couleurs que d'activités ─────────────────────────
import matplotlib.cm as cm
purpose_palette = [cm.tab20(i) for i in range(len(all_purposes))]

fig, ax = plt.subplots(figsize=(14, 8))
fig.patch.set_alpha(0)
ax.patch.set_alpha(0)

purpose_day[all_purposes].plot(
    kind="bar", stacked=True, ax=ax,
    color=purpose_palette, edgecolor="white", linewidth=0.5, alpha=0.88
)

for bar_idx, day in enumerate(day_order):
    cumulative = 0
    for purpose in all_purposes:
        val = purpose_day.loc[day, purpose] if purpose in purpose_day.columns else 0
        if val > 3 :
            ax.text(
                bar_idx,
                cumulative + val / 2,
                f"{val:.1f}%",
                ha="center", va="center",
                fontsize=10, color="black" #, fontweight="bold"
            )
        cumulative += val

ax.set_xlabel("Day of week", fontsize=12)
ax.set_ylabel("% of walk legs", fontsize=12)
ax.set_xticklabels(day_order, rotation=0, fontsize=12)
ax.tick_params(axis="y", labelsize=12)
ax.legend(title="Purpose", bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=12)
ax.axvspan(4.5, 6.5, alpha=0.08, color="orange")
ax.text(5.5, 102, "Weekend", ha="center", fontsize=12, color="orange")
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()
plt.close()

In [ ]:
purpose_group_day = (legs_GG_walk_regular
                     .dropna(subset=["purpose_group"])
                     .groupby(["day_of_week", "purpose_group"])
                     .size()
                     .unstack(fill_value=0)
                     .reindex(day_order)
                     .apply(lambda x: x / x.sum() * 100, axis=1))

cols_available = [g for g in PURPOSE_GROUP_ORDER if g in purpose_group_day.columns]

print("\n── Répartition par groupe de contrainte par jour (%) ────────────────")
print(purpose_group_day[cols_available].round(1).to_string())

print("\n── Walk purpose group distribution by day of week — stacked bar ─────")
fig, ax = plt.subplots(figsize=(8, 6))
fig.patch.set_alpha(0)
ax.patch.set_alpha(0)

purpose_group_day[cols_available].plot(
    kind="bar", stacked=True, ax=ax,
    color=[PURPOSE_GROUP_COLORS[g] for g in cols_available],
    edgecolor="white", linewidth=0.5, alpha=0.88
)

for bar_idx, day in enumerate(day_order):
    cumulative = 0
    for group in cols_available:
        val = purpose_group_day.loc[day, group] if day in purpose_group_day.index else 0
        if val > 3:
            ax.text(
                bar_idx,
                cumulative + val / 2,
                f"{val:.1f}%",
                ha="center", va="center",
                fontsize=10, color="black"
            )
        cumulative += val

#ax.set_title("Walk purpose group distribution by day of week (%)", fontweight="bold")
ax.set_xlabel("Day of week", fontsize=11)
ax.set_ylabel("% of walk legs", fontsize=12)
ax.set_xticklabels(day_order, rotation=0, fontsize=12)
ax.tick_params(axis="y", labelsize=12)
ax.axvspan(4.5, 6.5, alpha=0.08, color="orange")
ax.text(5.5, 101, "Weekend", ha="center", fontsize=12, color="orange")
ax.grid(axis="y", alpha=0.3)

ax.legend(
    handles=[mpatches.Patch(color=PURPOSE_GROUP_COLORS[g], label=g) for g in cols_available],
    loc="lower center",
    bbox_to_anchor=(0.5, -0.18),
    ncol=len(cols_available),
    fontsize=10,
    frameon=True
)

plt.tight_layout()
plt.show()
plt.close()

In [ ]:
# ─── Définir top_purposes ─────────────────────────────────────────────────────
# toutes les purposes présentes dans les données
top_purposes = (legs_GG_walk_regular["leading_stay_purpose"]
                .value_counts(dropna=True)
                .index.tolist())

print(f"top_purposes : {top_purposes}")

In [ ]:
# ─── Distance moyenne par trajet par destination ──────────────────────────────
distance_per_purpose = (legs_GG_walk_regular
                        .dropna(subset=["wgt_cant_trim_gps"])
                        .groupby(["user_id_fors", "leading_stay_purpose"])
                        .agg(
                            total_distance = ("length_weighted", "sum"),
                            n_trips        = ("leg_id",          "count")
                        )
                        .reset_index())

distance_per_purpose["mean_distance_per_trip"] = (
    distance_per_purpose["total_distance"] / distance_per_purpose["n_trips"]
)

# ─── Stats par destination ────────────────────────────────────────────────────
stats_by_purpose = (distance_per_purpose
                    .groupby("leading_stay_purpose")["mean_distance_per_trip"]
                    .agg(median="median", mean="mean", count="count",
                         min="min", max="max")
                    .reindex(top_purposes))

print("── Distance moyenne par trajet selon destination (m) — pondéré ──")
print(f"{'Destination':<20} {'N':>6} {'Médiane':>8} {'Moyenne':>8} {'Min':>8} {'Max':>8}")
print("-" * 62)
for p, row in stats_by_purpose.iterrows():
    if pd.notna(row['count']):
        print(f"{p:<20} {row['count']:>6.0f} {row['median']:>8.0f} {row['mean']:>8.0f} {row['min']:>8.0f} {row['max']:>8.0f}")

# ─── Plot ─────────────────────────────────────────────────────────────────────
data_by_purpose = [
    distance_per_purpose[
        distance_per_purpose["leading_stay_purpose"] == p
    ]["mean_distance_per_trip"].dropna().values
    for p in top_purposes
]

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle(
    "Distance moyenne par trajet selon la destination — pondéré\n"
    f"(users_GG_walk_regular — n={legs_GG_walk_regular['user_id_fors'].nunique()})",
    fontsize=13, fontweight="bold"
)

# ── Boxplot ────────────────────────────────────────────────────────────────
ax1 = axes[0]
bp = ax1.boxplot(
    data_by_purpose,
    labels=top_purposes,
    patch_artist=True,
    medianprops=dict(color="red", linewidth=2),
    flierprops=dict(marker="o", markersize=2, alpha=0.3)
)
for patch, color in zip(bp["boxes"], PALETTE[:len(top_purposes)]):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)

ax1.set_yscale("log")
ax1.set_title("Distribution de la distance par trajet", fontweight="bold")
ax1.set_xlabel("Destination")
ax1.set_ylabel("Distance (m) — échelle log")
ax1.tick_params(axis="x", rotation=30)
ax1.grid(axis="y", alpha=0.3, which="both")

# ── Barplot médiane + IQR ──────────────────────────────────────────────────
ax2 = axes[1]

medians = stats_by_purpose["median"].values
q25 = [distance_per_purpose[distance_per_purpose["leading_stay_purpose"] == p]["mean_distance_per_trip"].quantile(0.25)
       for p in top_purposes]
q75 = [distance_per_purpose[distance_per_purpose["leading_stay_purpose"] == p]["mean_distance_per_trip"].quantile(0.75)
       for p in top_purposes]

x      = np.arange(len(top_purposes))
colors = PALETTE[:len(top_purposes)]

bars = ax2.bar(x, medians,
               color=colors, edgecolor='white', alpha=0.85,
               yerr=[medians - np.array(q25), np.array(q75) - medians],
               error_kw=dict(ecolor='black', capsize=3, linewidth=0.8))

for bar, (p, row) in zip(bars, stats_by_purpose.iterrows()):
    if pd.notna(row['count']):
        ax2.text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + max(np.array(q75) - medians) * 0.05,
                 f"n={int(row['count'])}", ha='center', fontsize=7)

ax2.set_xticks(x)
ax2.set_xticklabels(top_purposes, rotation=30, ha='right', fontsize=8)
ax2.set_title("Médiane + IQR de la distance par trajet", fontweight="bold")
ax2.set_ylabel("Distance médiane (m)")
ax2.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ─── Distance moyenne par trajet par destination ──────────────────────────────
distance_per_purpose = (legs_GG_walk_regular
                        .dropna(subset=["wgt_cant_trim_gps"])
                        .groupby(["user_id_fors", "leading_stay_purpose"])
                        .agg(
                            total_distance = ("length_weighted", "sum"),
                            n_trips        = ("leg_id",          "count")
                        )
                        .reset_index())

distance_per_purpose["mean_distance_per_trip"] = (
    distance_per_purpose["total_distance"] / distance_per_purpose["n_trips"]
)

# ─── Stats par destination ────────────────────────────────────────────────────
stats_by_purpose = (distance_per_purpose
                    .groupby("leading_stay_purpose")["mean_distance_per_trip"]
                    .agg(median="median", mean="mean", count="count",
                         min="min", max="max")
                    .reindex(top_purposes))

print("── Distance moyenne par trajet selon destination (m) — pondéré ──")
print(f"{'Destination':<20} {'N':>6} {'Médiane':>8} {'Moyenne':>8} {'Min':>8} {'Max':>8}")
print("-" * 62)
for p, row in stats_by_purpose.iterrows():
    if pd.notna(row['count']):
        print(f"{p:<20} {row['count']:>6.0f} {row['median']:>8.0f} {row['mean']:>8.0f} {row['min']:>8.0f} {row['max']:>8.0f}")

# ─── Couleurs par activité basées sur le groupe de contrainte ─────────────────
purpose_colors = [
    PURPOSE_GROUP_COLORS.get(PURPOSE_GROUP_MAP.get(p, "Other"), "#8C8C8C")
    for p in top_purposes
]

# ─── Boxplot ─────────────────────────────────────────────────────────────────
print("\n── Mean walk distance per trip by purpose — boxplot ─────────────────")
fig1, ax1 = plt.subplots(figsize=(12, 6))
fig1.patch.set_alpha(0)
ax1.patch.set_alpha(0)

bp = ax1.boxplot(
    data_by_purpose,
    labels=top_purposes,
    patch_artist=True,
    medianprops=dict(color="red", linewidth=2),
    flierprops=dict(marker="o", markersize=2, alpha=0.3)
)
for patch, color in zip(bp["boxes"], purpose_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)

ax1.set_yscale("log")
#ax1.set_title("Distribution of mean walk distance per trip by purpose", fontweight="bold")
ax1.set_xlabel("Destination", fontsize=11)
ax1.set_ylabel("Distance (m) — log scale", fontsize=11)
ax1.tick_params(axis="x", rotation=30, labelsize=10)
ax1.tick_params(axis="y", labelsize=10)
ax1.grid(axis="y", alpha=0.3, which="both")
ax1.legend(handles=[mpatches.Patch(color=PURPOSE_GROUP_COLORS[g], label=g)
                    for g in PURPOSE_GROUP_ORDER],
           fontsize=9, loc="upper right")

plt.tight_layout()
plt.show()
plt.close()


# ─── Barplot médiane + IQR ────────────────────────────────────────────────────
print("\n── Mean walk distance per trip by purpose — median + IQR barplot ────")
fig2, ax2 = plt.subplots(figsize=(9, 6))
fig2.patch.set_alpha(0)
ax2.patch.set_alpha(0)

medians = stats_by_purpose["median"].values
q25 = [distance_per_purpose[distance_per_purpose["leading_stay_purpose"] == p]["mean_distance_per_trip"].quantile(0.25)
       for p in top_purposes]
q75 = [distance_per_purpose[distance_per_purpose["leading_stay_purpose"] == p]["mean_distance_per_trip"].quantile(0.75)
       for p in top_purposes]

x = np.arange(len(top_purposes))

bars = ax2.bar(x, medians,
               color=purpose_colors, edgecolor="white", alpha=0.85,
               yerr=[medians - np.array(q25), np.array(q75) - medians],
               error_kw=dict(ecolor="black", capsize=3, linewidth=0.8))

for bar, (p, row) in zip(bars, stats_by_purpose.iterrows()):
    if pd.notna(row["count"]):
        ax2.text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() * 0.05,  # 5% de la hauteur de la barre
                 f"n={int(row['count'])}",
                 ha="center", va="bottom", fontsize=8,
                 color="black") #fontweight="bold"

ax2.set_xticks(x)
ax2.set_xticklabels(top_purposes, rotation=30, ha="right", fontsize=10)
#ax2.set_title("Median + IQR of mean walk distance per trip by purpose", fontweight="bold")
ax2.set_ylabel("Median distance (m)", fontsize=11)
ax2.tick_params(axis="y", labelsize=10)
ax2.grid(axis="y", alpha=0.3)
ax2.legend(handles=[mpatches.Patch(color=PURPOSE_GROUP_COLORS[g], label=g)
                    for g in PURPOSE_GROUP_ORDER],
           fontsize=9, loc="upper right")

plt.tight_layout()
plt.show()
plt.close()

In [ ]:
# ─── Distance moyenne par trajet par groupe de contrainte ─────────────────────
distance_per_group = (legs_GG_walk_regular
                      .dropna(subset=["wgt_cant_trim_gps", "purpose_group"])
                      .groupby(["user_id_fors", "purpose_group"])
                      .agg(
                          total_distance = ("length_weighted", "sum"),
                          n_trips        = ("leg_id",          "count")
                      )
                      .reset_index())

distance_per_group["mean_distance_per_trip"] = (
    distance_per_group["total_distance"] / distance_per_group["n_trips"]
)

stats_by_group = (distance_per_group
                  .groupby("purpose_group")["mean_distance_per_trip"]
                  .agg(median="median", mean="mean", count="count")
                  .reindex(PURPOSE_GROUP_ORDER))

print("── Distance médiane par trajet selon groupe de contrainte (m) ──")
print(stats_by_group)

# ─── Barplot médiane + IQR ────────────────────────────────────────────────────
print("\n── Mean walk distance per trip by constraint group — median + IQR ───")
fig, ax = plt.subplots(figsize=(5, 4))
fig.patch.set_alpha(0)
ax.patch.set_alpha(0)

groups    = [g for g in PURPOSE_GROUP_ORDER if g in stats_by_group.index]
medians   = stats_by_group.loc[groups, "median"].values
q25 = [distance_per_group[distance_per_group["purpose_group"] == g]["mean_distance_per_trip"].quantile(0.25)
       for g in groups]
q75 = [distance_per_group[distance_per_group["purpose_group"] == g]["mean_distance_per_trip"].quantile(0.75)
       for g in groups]

x      = np.arange(len(groups))
colors = [PURPOSE_GROUP_COLORS[g] for g in groups]

bars = ax.bar(x, medians,
              color=colors, edgecolor="white", alpha=0.85,
              yerr=[medians - np.array(q25), np.array(q75) - medians],
              error_kw=dict(ecolor="black", capsize=4, linewidth=1))

# ─── Valeurs médianes à droite de la barre IQR ───────────────────────────────
for i, (median, q75_val) in enumerate(zip(medians, q75)):
    ax.text(x[i] + 0.10,          # décalage à droite de la barre
            median + 10,                # au niveau de la médiane
            f"{median:.0f}m",
            ha="left", va="center", fontsize=9, color="black")

# for bar, g in zip(bars, groups):
#     n = int(stats_by_group.loc[g, "count"])
#     ax.text(bar.get_x() + bar.get_width()/2,
#             bar.get_height() * 0.05,
#             f"n={n:,}",
#             ha="center", va="bottom", fontsize=9,
#             color="black") #fontweight="bold"

ax.set_xticks(x)
ax.set_xticklabels(groups, rotation=0, fontsize=11)
#ax.set_title("Median walk distance per trip by constraint group", fontweight="bold")
ax.set_ylabel("Median distance (m)", fontsize=11)
ax.tick_params(axis="y", labelsize=10)
ax.tick_params(axis="x", labelsize=10, rotation=35)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()
plt.close()

In [ ]:
distance_per_purpose

### TRIPS and JOURNEYS

In [ ]:
# ─── Load trips and journeys CSV ──────────────────────────────────────────────
trips    = pd.read_csv(f'{input_file_path_PL}241120_trips.csv')
journeys = pd.read_csv(f'{input_file_path_PL}241120_journey.csv')

print(f"Trips    : {len(trips)}")
print(f"Journeys : {len(journeys)}")

# ─── Quick overview ───────────────────────────────────────────────────────────
print("\n── Trip modes ───────────────────────────────────────")
print(trips["mode"].value_counts())

print("\n── Trip purposes ────────────────────────────────────")
print(trips["purpose"].value_counts())

print("\n── Journey main modes ───────────────────────────────")
print(journeys["main_mode"].value_counts())

print("\n── Journey main purposes ────────────────────────────")
print(journeys["main_purpose"].value_counts())

In [ ]:
trips.head()

In [ ]:
trips.dtypes

In [ ]:
journeys.head()

In [ ]:
journeys.dtypes

In [ ]:
# ─── Filter walk-only trips ───────────────────────────────────────────────────
# Check what walk looks like in mode column first
print(trips["mode"].unique())

# ─── Trips where main mode is walk ───────────────────────────────────────────
walk_trips = trips[trips["mode"] == "walk"]  # adapt value if different

print(f"\nTotal trips           : {len(trips)}")
print(f"Walk-only trips       : {len(walk_trips)}")
print(f"% walk trips          : {len(walk_trips)/len(trips)*100:.1f}%")

# ─── Purpose distribution of walk-only trips ──────────────────────────────────
print("\n── Purpose of mainly walk trips ───────────────────────")
purpose_walk = walk_trips["purpose"].value_counts()
print(purpose_walk)

# ─── Users who mainly walk (main trips = walk) ───────────────────────────
trips_per_user_mode = (trips
                       .groupby("user_id_fors")["mode"]
                       .apply(lambda x: x.unique().tolist())
                       .reset_index()
                       .rename(columns={"mode": "modes_used"}))

walk_mainly_users = trips_per_user_mode[
    trips_per_user_mode["modes_used"].apply(lambda x: x == ["walk"])
]

print(f"\n── Users who ONLY use walk for all trips ────────────")
print(f"Walk-mainly users : {len(walk_mainly_users)} / {trips['user_id_fors'].nunique()}")

# ─── What purposes do walk-only users go to ? ────────────────────────────────
walk_mainly_trips = trips[trips["user_id_fors"].isin(walk_mainly_users["user_id_fors"])]
print("\n── Purposes of walk-only users ──────────────────────")
print(walk_mainly_trips["purpose"].value_counts())

# ─── Merge with user profile ──────────────────────────────────────────────────
walk_only_profile = walk_mainly_users.merge(
    user_stat[["user_id_fors", "gdr", "age_fr", "prof"]],
    on="user_id_fors",
    how="left"
)

print("\n── Profile of walk-mainly users ───────────────────────")
for col in ["gdr", "age_fr", "prof"]:
    print(f"\n  {col}:")
    print(walk_only_profile[col].value_counts())

In [ ]:
# Users where more than 50% of their trips are walk
walk_ratio_per_user = (trips
                       .groupby("user_id_fors")["mode"]
                       .apply(lambda x: (x == "walk").sum() / len(x))
                       .reset_index()
                       .rename(columns={"mode": "walk_ratio"}))

# Different thresholds
for threshold in [0.5, 0.7, 0.9, 1.0]:
    n = (walk_ratio_per_user["walk_ratio"] >= threshold).sum()
    print(f"Users with >{threshold*100:.0f}% walk trips : {n}")

### CAR IN THE HOUSEOLD

In [ ]:
# ─── Convert to numeric ───────────────────────────────────────────────────────
users_GG_walk_regular["car_in_HH_count"] = pd.to_numeric(
    users_GG_walk_regular["car_in_HH_count"], errors="coerce"
)

# ─── Quick overview ───────────────────────────────────────────────────────────
print("── Car in Household ─────────────────────────────────")
print(f"Total users   : {len(users_GG_walk_regular)}")
print(f"NaN           : {users_GG_walk_regular['car_in_HH_count'].isna().sum()}")
print(f"\nValue counts  :")

n_valid = users_GG_walk_regular["car_in_HH_count"].notna().sum()
vc      = users_GG_walk_regular["car_in_HH_count"].value_counts().sort_index()

for val, cnt in vc.items():
    pct = cnt / n_valid * 100
    print(f"   {val} car(s)  : {cnt:>5}  ({pct:.1f}%)")

# ─── Stats by age group ───────────────────────────────────────────────────────
print("\n── Mean cars per HH by age group ────────────────────")
print(users_GG_walk_regular.groupby("age_fr_grouped")["car_in_HH_count"]
      .agg(mean="mean", median="median", count="count")
      .round(2))

# ─── Stats by social situation ────────────────────────────────────────────────
print("\n── Mean cars per HH by social situation ─────────────")
print(users_GG_walk_regular.groupby("prof")["car_in_HH_count"]
      .agg(mean="mean", median="median", count="count")
      .round(2))

# ─── Plot ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Number of Cars per Household — Geneva Walkers",
             fontsize=14, fontweight="bold")

PALETTE = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2"]

# ─── Chart 1 : by age group ───────────────────────────────────────────────────
ax1 = axes[0]

age_order = ["18-29 ans", "30-44 ans", "45-59 ans", "60 ans +"]
car_by_age = (users_GG_walk_regular
              .groupby("age_fr_grouped")["car_in_HH_count"]
              .value_counts(normalize=True)
              .mul(100)
              .unstack(fill_value=0)
              .reindex(age_order))

car_by_age.plot(kind="bar", stacked=True, ax=ax1,
                colormap="viridis", edgecolor="white",
                linewidth=0.5, alpha=0.88)

ax1.set_title("Car ownership distribution by age group", fontweight="bold")
ax1.set_xlabel("Age group")
ax1.set_ylabel("% of users")
ax1.set_xticklabels(age_order, rotation=30, ha="right")
ax1.legend(title="N° of cars", bbox_to_anchor=(1.01, 1),
           loc="upper left", fontsize=8)
ax1.grid(axis="y", alpha=0.3)

# ─── Chart 2 : by social situation ───────────────────────────────────────────
ax2 = axes[1]

car_by_prof = (users_GG_walk_regular
               .groupby("prof")["car_in_HH_count"]
               .value_counts(normalize=True)
               .mul(100)
               .unstack(fill_value=0))

car_by_prof.plot(kind="bar", stacked=True, ax=ax2,
                 colormap="viridis", edgecolor="white",
                 linewidth=0.5, alpha=0.88)

ax2.set_title("Car ownership distribution by social situation", fontweight="bold")
ax2.set_xlabel("Social situation")
ax2.set_ylabel("% of users")
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=30, ha="right")
ax2.legend(title="N° of cars", bbox_to_anchor=(1.01, 1),
           loc="upper left", fontsize=8)
ax2.grid(axis="y", alpha=0.3)

plt.tight_layout()
#plt.savefig("car_ownership_by_profile.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_alpha(0)
ax.patch.set_alpha(0)

age_order = [label for _, label in age_filters_en]  # ← corrigé

car_by_age = (users_GG_walk_regular
              .dropna(subset=["age_fr_grouped"])
              .assign(age_en=lambda df: df["age_fr_grouped"].map(age_fr_to_en))  # ← lambda
              .groupby("age_en")["car_in_HH_count"]
              .value_counts(normalize=True)
              .mul(100)
              .unstack(fill_value=0)
              .reindex(age_order))

car_by_age.plot(kind="bar", stacked=True, ax=ax,
                colormap="viridis", edgecolor="white",
                linewidth=0.5, alpha=0.88)

# ─── % dans les barres ────────────────────────────────────────────────────────
for i, age in enumerate(car_by_age.index):
    cumul = 0
    for col in car_by_age.columns:
        val = car_by_age.loc[age, col]
        if val > 5:  # ← seuil minimum pour afficher
            ax.text(i, cumul + val / 2, f"{val:.0f}%",
                    ha="center", va="center", fontsize=11, color="white", fontweight="bold")
        cumul += val

#ax.set_title("Car ownership distribution by age group", fontweight="bold", fontsize=14)
#ax.set_xlabel("Age group", fontsize=13)
ax.set_ylabel("% of users", fontsize=13)
ax.set_xticklabels(age_order, rotation=30, ha="right", fontsize=12)
ax.tick_params(axis='y', labelsize=12)
ax.legend(title="Number \n of cars", bbox_to_anchor=(1.01, 1),
          loc="upper left", fontsize=11, title_fontsize=11)
ax.grid(axis="y", alpha=0.3)
ax.set_xlabel("")
plt.tight_layout()
plt.show()

In [ ]:
car_by_age = (users_GG_walk_regular
              .dropna(subset=["age_fr_grouped"])
              .assign(age_en=lambda df: df["age_fr_grouped"].map(age_fr_to_en))
              .groupby("age_en")["car_in_HH_count"]
              .value_counts(normalize=True)
              .mul(100)
              .unstack(fill_value=0)
              .reindex(age_order))

print(car_by_age)

## ETUDE SOCIO-DEMOGRAPHIQUE

In [ ]:
PALETTE = ["#4C72B0", "#DD8452", "#55A868", "#C44E52",
           "#8172B2", "#937860", "#DA8BC3", "#8C8C8C",
           "#64B5CD", "#B0C4DE", "#FFD700"]

# ─── Ordres extraits directement depuis les labels déjà définis ───────────────
tp_abo_order       = list(tp_labels_Q10.values())
mobility_abo_order = list(tp_labels_Q11.values())

# ─── Mappings à appliquer à la volée ─────────────────────────────────────────
col_mappings = {
    'gdr'            : gender_fr_to_en,
    'age_fr_grouped' : age_fr_to_en,
    'tp_level'       : tp_level_map,
}

# ─── Définir les facteurs à analyser ─────────────────────────────────────────
factors = {
    'gdr'            : ('Gender',                      None),
    'age_fr_grouped' : ('Age group',                   None),
    'income_class'   : ('Income per earner (ref. GE)', income_class_order),
    'has_car'        : ('Car ownership',               [labels_all_groups["no_car"], labels_all_groups["has_car"]]),
    'tp_level'       : ('PT subscription level',       tp_level_order),
}

# ─── Définir les facteurs à analyser ──────────────────────────────────────────
factors = {
    'gdr'                 : ('Gender',                           None),
    'age_fr_grouped'      : ('Age group',                        None),
    #'prof'                : ('Socio-professional situation',      None),
    #'Q120_label'          : ('Household income (CHF/month)',      income_order_Q120),
    'income_class'     : ('Income per earner (ref. GE)',        income_class_order),
    #'has_car'             : ('Car ownership',                     ['No car', 'With car']),
    'has_car'             : ('Car ownership',                   [labels_all_groups["no_car"], labels_all_groups["has_car"]]),
    #'main_tp_abo'         : ('Main PT subscription',              tp_abo_order),
    'tp_level' : ('PT subscription level',                tp_level_order),
    #'main_mobility_abo'   : ('Main mobility subscription',        mobility_abo_order),
    #'mobility_level'      : ('Mobility subscription level',       [0, 1, 2]),
    #'walk_intensity'      : ('Walking intensity',                 walk_intensity_order),
    #'physical_cond_label' : ('Perceived physical health',         physical_cond_label_order),
}

#DIST_VAR       = 'dist_walk_per_walk_day'
DIST_VAR       = 'dist_walk_per_day'
#DIST_VAR_LABEL = 'Distance walked per walking day (m)'
DIST_VAR_LABEL = 'Distance walked per day detected in GE (m)'

# ─── Labels sur users_GG_walk_regular ────────────────────────────────────────
users_GG_walk_regular['Q120_label'] = users_GG_walk_regular['Q120'].map(income_labels_Q120)
"""users_GG_walk_regular['has_car']    = users_GG_walk_regular['car_in_HH_count'].apply(
    lambda x: 'With car' if pd.notna(x) and x > 0 else ('No car' if pd.notna(x) else np.nan)
)"""

#users_GG_walk_regular["tp_level"] = users_GG_walk_regular["tp_level"].map(tp_level_map)

#─── Stats texte ──────────────────────────────────────────────────────────────
print("=" * 65)
print("  AVERAGE WALKING DISTANCE BY SOCIO-DEMOGRAPHIC FACTOR")
print("=" * 65)

for col, (label, order) in factors.items():
    _data_txt = users_GG_walk_regular.dropna(subset=[DIST_VAR]).copy()

    # ─── Mapping à la volée si nécessaire ────────────────────────────────
    if col in col_mappings:
        _data_txt[col] = _data_txt[col].map(col_mappings[col])

    _data_txt = _data_txt.dropna(subset=[col]).copy()

    if order is not None:
        order_filtered = [o for o in order if o in _data_txt[col].dropna().values]
        _data_txt[col] = pd.Categorical(_data_txt[col], categories=order_filtered, ordered=True)

    stats = (_data_txt
             .groupby(col)[DIST_VAR]
             .agg(n='count', median='median', mean='mean',
                  q25=lambda x: x.quantile(0.25),
                  q75=lambda x: x.quantile(0.75))
             .reset_index()
             .sort_values(col))

    print(f"\n── {label} ──────────────────────────────────────")
    print(stats[[col, 'n', 'median', 'mean', 'q25', 'q75']].to_string(index=False))

# ─── Graphiques séparés par facteur ──────────────────────────────────────────
for row_idx, (col, (label, order)) in enumerate(factors.items()):

    _data = users_GG_walk_regular.dropna(subset=[DIST_VAR]).copy()

    # ─── Mapping à la volée si nécessaire ────────────────────────────────
    if col in col_mappings:
        _data[col] = _data[col].map(col_mappings[col])

    _data   = _data.dropna(subset=[col]).copy()
    n_valid = len(_data)

    # ── Ordre dynamique ───────────────────────────────────────────────────
    if order is not None:
        order_filtered = [o for o in order if o in _data[col].dropna().values]
        _data[col]  = pd.Categorical(_data[col], categories=order_filtered, ordered=True)
        categories  = order_filtered
    else:
        categories = sorted(_data[col].dropna().unique())

    stats = (_data.groupby(col)[DIST_VAR]
             .agg(n='count', median='median',
                  q25=lambda x: x.quantile(0.25),
                  q75=lambda x: x.quantile(0.75))
             .reindex(categories))

    colors = PALETTE[:len(categories)]
    x      = np.arange(len(categories))

    # ── Barplot médiane + IQR ─────────────────────────────────────────────
    print(f"\n── {label} (n={n_valid}) — median + IQR ────────────────────────")
    fig, ax_bar = plt.subplots(figsize=(4, 5))
    fig.patch.set_alpha(0)
    ax_bar.patch.set_alpha(0)

    bars = ax_bar.bar(x, stats['median'],
                      color=colors, edgecolor='white', alpha=0.85,
                      yerr=[stats['median'] - stats['q25'],
                            stats['q75']    - stats['median']],
                      error_kw=dict(ecolor='black', capsize=3, linewidth=0.8))

    # for bar, (cat, row_s) in zip(bars, stats.iterrows()):
    #     if pd.notna(row_s['n']):
    #         ax_bar.text(bar.get_x() + bar.get_width()/2,
    #                     bar.get_height() + (stats['q75'].max() * 0.02),
    #                     f"n={int(row_s['n'])}", ha='center', fontsize=9)

    ax_bar.set_xticks(x)
    ax_bar.set_xticklabels([str(c) for c in categories], rotation=30, ha='right', fontsize=10)
    ax_bar.set_ylabel(f'{DIST_VAR_LABEL} — median', fontsize=10)
    ax_bar.tick_params(axis='y', labelsize=10)
    #ax_bar.set_title(f"{label} (n={n_valid}) — median + IQR", fontweight='bold')  # commenter pour export rapport
    ax_bar.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.show()
    plt.close()

    # ── Boxplot ───────────────────────────────────────────────────────────
    print(f"\n── {label} (n={n_valid}) — distribution ────────────────────────")
    fig, ax_box = plt.subplots(figsize=(4, 5))
    fig.patch.set_alpha(0)
    ax_box.patch.set_alpha(0)

    data_by_cat = [_data[_data[col] == cat][DIST_VAR].dropna().values
                   for cat in categories]

    bp = ax_box.boxplot(
        data_by_cat,
        labels=[str(c) for c in categories],
        patch_artist=True,
        medianprops=dict(color='red', linewidth=2),
        flierprops=dict(marker='o', markersize=3, alpha=0.3),
        widths=0.5
    )
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.75)

    ax_box.set_yscale('log')
    ax_box.set_xticklabels([str(c) for c in categories], rotation=30, ha='right', fontsize=10)
    ax_box.set_ylabel(f'{DIST_VAR_LABEL} — log scale', fontsize=10)
    ax_box.tick_params(axis='y', labelsize=10)
    #ax_box.set_title(f"{label} (n={n_valid}) — distribution", fontweight='bold')  # commenter pour export rapport
    ax_box.grid(axis='y', alpha=0.3, which='both')

    plt.tight_layout()
    plt.show()
    plt.close()

In [ ]:
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

PALETTE = ["#4C72B0", "#DD8452", "#55A868", "#C44E52",
           "#8172B2", "#937860", "#DA8BC3", "#8C8C8C",
           "#64B5CD", "#B0C4DE", "#FFD700"]

# ─── Ordres extraits directement depuis les labels déjà définis ───────────────
tp_abo_order       = list(tp_labels_Q10.values())
mobility_abo_order = list(tp_labels_Q11.values())

# ─── Mappings à appliquer à la volée ─────────────────────────────────────────
col_mappings = {
    'gdr'            : gender_fr_to_en,
    'age_fr_grouped' : age_fr_to_en,
    'tp_level'       : tp_level_map,
}

# ─── Définir les facteurs à analyser ─────────────────────────────────────────
factors = {
    'gdr'            : ('Gender',                      None),
    'age_fr_grouped' : ('Age group',                   None),
    'income_class'   : ('Income per earner (ref. GE)', income_class_order),
    'has_car'        : ('Car ownership',               [labels_all_groups["no_car"], labels_all_groups["has_car"]]),
    'tp_level'       : ('PT subscription level',       tp_level_order),
}

# ─── Définir les facteurs à analyser ──────────────────────────────────────────
factors = {
    'gdr'                 : ('Gender',                           None),
    'age_fr_grouped'      : ('Age group',                        None),
    #'prof'                : ('Socio-professional situation',      None),
    #'Q120_label'          : ('Household income (CHF/month)',      income_order_Q120),
    'income_class'     : ('Income per earner (ref. GE)',        income_class_order),
    #'has_car'             : ('Car ownership',                     ['No car', 'With car']),
    'has_car'             : ('Car ownership',                   [labels_all_groups["no_car"], labels_all_groups["has_car"]]),
    #'main_tp_abo'         : ('Main PT subscription',              tp_abo_order),
    'tp_level' : ('PT subscription level',                tp_level_order),
    #'main_mobility_abo'   : ('Main mobility subscription',        mobility_abo_order),
    #'mobility_level'      : ('Mobility subscription level',       [0, 1, 2]),
    #'walk_intensity'      : ('Walking intensity',                 walk_intensity_order),
    #'physical_cond_label' : ('Perceived physical health',         physical_cond_label_order),
}

#DIST_VAR       = 'dist_walk_per_walk_day'
DIST_VAR       = 'dist_walk_per_day'
#DIST_VAR_LABEL = 'Distance walked per walking day (m)'
DIST_VAR_LABEL = 'Distance (m)'

# ─── Labels sur users_GG_walk_regular ────────────────────────────────────────
users_GG_walk_regular['Q120_label'] = users_GG_walk_regular['Q120'].map(income_labels_Q120)
"""users_GG_walk_regular['has_car']    = users_GG_walk_regular['car_in_HH_count'].apply(
    lambda x: 'With car' if pd.notna(x) and x > 0 else ('No car' if pd.notna(x) else np.nan)
)"""

#users_GG_walk_regular["tp_level"] = users_GG_walk_regular["tp_level"].map(tp_level_map)

#─── Stats texte ──────────────────────────────────────────────────────────────
print("=" * 65)
print("  AVERAGE WALKING DISTANCE BY SOCIO-DEMOGRAPHIC FACTOR")
print("=" * 65)

for col, (label, order) in factors.items():
    _data_txt = users_GG_walk_regular.dropna(subset=[DIST_VAR]).copy()

    if col in col_mappings:
        _data_txt[col] = _data_txt[col].map(col_mappings[col])

    _data_txt = _data_txt.dropna(subset=[col]).copy()

    if order is not None:
        order_filtered = [o for o in order if o in _data_txt[col].dropna().values]
        _data_txt[col] = pd.Categorical(_data_txt[col], categories=order_filtered, ordered=True)

    stats = (_data_txt
             .groupby(col)[DIST_VAR]
             .agg(n='count', median='median', mean='mean',
                  q25=lambda x: x.quantile(0.25),
                  q75=lambda x: x.quantile(0.75))
             .reset_index()
             .sort_values(col))

    # ─── % par rapport à la première catégorie ────────────────────────────
    ref_median = stats['median'].iloc[0]
    stats['vs_ref (%)'] = ((stats['median'] - ref_median) / ref_median * 100).round(1)
    stats['vs_ref (%)'] = stats['vs_ref (%)'].apply(lambda x: f"+{x}%" if x > 0 else f"{x}%")
    stats.loc[stats.index[0], 'vs_ref (%)'] = "ref."

    print(f"\n── {label} ──────────────────────────────────────")
    print(stats[[col, 'n', 'median', 'mean', 'q25', 'q75', 'vs_ref (%)']].to_string(index=False))

# ─── Graphiques séparés par facteur ──────────────────────────────────────────
for row_idx, (col, (label, order)) in enumerate(factors.items()):

    _data = users_GG_walk_regular.dropna(subset=[DIST_VAR]).copy()

    if col in col_mappings:
        _data[col] = _data[col].map(col_mappings[col])

    _data   = _data.dropna(subset=[col]).copy()
    n_valid = len(_data)

    if order is not None:
        order_filtered = [o for o in order if o in _data[col].dropna().values]
        _data[col]  = pd.Categorical(_data[col], categories=order_filtered, ordered=True)
        categories  = order_filtered
    else:
        categories = sorted(_data[col].dropna().unique())

    stats = (_data.groupby(col)[DIST_VAR]
             .agg(n='count', median='median',
                  q25=lambda x: x.quantile(0.25),
                  q75=lambda x: x.quantile(0.75))
             .reindex(categories))

    colors = PALETTE[:len(categories)]
    y      = np.arange(len(categories))  # ← axe horizontal maintenant

    # ── Dot plot ──────────────────────────────────────────────────────────
    print(f"\n── {label} (n={n_valid}) — dot plot ────────────────────────────")
    fig_width  = 3.5  # fixe pour tous
    fig_height = 5    # fixe pour tous
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    fig.patch.set_alpha(0)
    ax.patch.set_alpha(0)
    ax.set_xlim(-0.5, len(categories) - 0.5)

    # ─── Ligne IQR ───────────────────────────────────────────────────────
    for i, (cat, row_s) in enumerate(stats.iterrows()):
        ax.plot([i, i], [row_s['q25'], row_s['q75']],
                color=colors[i], linewidth=2, alpha=0.6)

    # ─── Point médiane ────────────────────────────────────────────────────
    ax.scatter(y, stats['median'],
               color=colors, s=100, zorder=5, edgecolors='white', linewidth=0.8)

    ax.set_xticks(y)
    ax.set_xticklabels([str(c) for c in categories], rotation=30, ha='right', fontsize=10)
    ax.set_ylabel(f'{DIST_VAR_LABEL} — median', fontsize=10)
    ax.tick_params(axis='y', labelsize=10)
    #ax.set_title(f"{label} (n={n_valid})", fontweight='bold')
    ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.show()
    plt.close()

In [ ]:
# ─── Modes à afficher et leurs couleurs ───────────────────────────────────────
mode_display = {
    "walk"                     : ("Walk",           "#4C72B0"),
    "car"                      : ("Car",            "#DD8452"),
    "public_transit"           : ("Public Transit", "#55A868"),
    "train"                    : ("Train",          "#C44E52"),
    "bicycle"                  : ("Bicycle",        "#8172B2"),
    "vehicle_equivalent_device": ("Micro-mobility", "#937860"),
    "2WV"                      : ("2-wheel vehicle","#DA8BC3"),
    "other"                    : ("Other",          "#8C8C8C"),
    "boat"                     : ("Boat",           "#64B5CD"),
    "airplane"                 : ("Airplane",       "#B0C4DE"),
}

# ─── Facteurs socio-démographiques ────────────────────────────────────────────
factors = {
    'gdr'             : ('Genre',                      None),
    'age_fr_grouped'          : ("Tranche d'âge",              None),
    'income_class' : ('Revenu par actif (réf. GE)', income_class_order),
    'has_car'         : ('Possession voiture',         ['Sans voiture', 'Avec voiture']),
}

# ─── Déduplication : une ligne par user ───────────────────────────────────────
user_cols = (
    ["user_id_fors"] +
    list(factors.keys()) +
    [f"share_dist_{m}" for m in mode_display.keys()] +
    [f"share_legs_{m}" for m in mode_display.keys()]
)
user_cols_existing = [c for c in user_cols if c in users_GG_walk_regular.columns]
df_users = users_GG_walk_regular[user_cols_existing].drop_duplicates("user_id_fors")

print(f"Users uniques : {len(df_users)}")

# ─── Fonction plot stacked bar ─────────────────────────────────────────────────
def plot_modal_share_stacked(df, factor_col, factor_label, order,
                              metric_prefix, metric_title):
    
    _data = df.dropna(subset=[factor_col]).copy()

    if order is not None:
        order_filtered = [o for o in order if o in _data[factor_col].dropna().values]
        _data[factor_col] = pd.Categorical(_data[factor_col],
                                           categories=order_filtered, ordered=True)
        categories = order_filtered
    else:
        categories = sorted(_data[factor_col].dropna().unique())

    # ─── Calculer la part modale moyenne par groupe ───────────────────────────
    share_cols = {
        mode: f"{metric_prefix}_{mode}"
        for mode in mode_display.keys()
        if f"{metric_prefix}_{mode}" in _data.columns
    }

    means = {}
    ns    = {}
    for cat in categories:
        subset     = _data[_data[factor_col] == cat]
        ns[cat]    = len(subset)
        means[cat] = {
            mode: subset[col].mean() * 100
            for mode, col in share_cols.items()
            if col in subset.columns
        }

    # ─── Résumé textuel ───────────────────────────────────────────────────────
    print(f"\n{'═'*70}")
    print(f"  {metric_title} — {factor_label}")
    print(f"{'═'*70}")

    # ─── Header ───────────────────────────────────────────────────────────────
    mode_labels_display = {
        mode: label
        for mode, (label, _) in mode_display.items()
        if mode in share_cols
    }
    header = f"  {'Groupe':<25} {'n':>5}"
    for mode_label in mode_labels_display.values():
        header += f"  {mode_label:>12}"
    print(header)
    print(f"  {'-'*25}-{'-'*5}" + f"--{'-'*12}" * len(mode_labels_display))

    for cat in categories:
        line = f"  {str(cat):<25} {ns[cat]:>5}"
        for mode in mode_labels_display.keys():
            val = means[cat].get(mode, 0)
            line += f"  {val:>11.1f}%"
        print(line)

    # ─── Plot ─────────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(max(8, len(categories) * 1.5), 6))
    fig.suptitle(
        f"{metric_title} — {factor_label}\n"
        f"(users_GG_walk_regular, moyenne par groupe)",
        fontsize=13, fontweight="bold"
    )

    x      = np.arange(len(categories))
    bottom = np.zeros(len(categories))

    for mode, (mode_label, color) in mode_display.items():
        if mode not in share_cols:
            continue
        values = np.array([means[cat].get(mode, 0) for cat in categories])

        if values.mean() < 0.5:
            continue

        bars = ax.bar(x, values, bottom=bottom, color=color,
                      edgecolor="white", linewidth=0.5, label=mode_label)

        for i, (val, bot) in enumerate(zip(values, bottom)):
            if val > 3:
                ax.text(x[i], bot + val / 2, f"{val:.1f}%",
                        ha='center', va='center', fontsize=7,
                        color='white', fontweight='bold')

        bottom += values

    ax.set_xticks(x)
    ax.set_xticklabels(
        [f"{cat}\n(n={ns[cat]})" for cat in categories],
        rotation=30, ha='right', fontsize=9
    )
    ax.set_ylabel("Part modale moyenne (%)")
    ax.set_ylim(0, 105)
    ax.grid(axis='y', alpha=0.3)
    ax.legend(loc='upper right', fontsize=8, bbox_to_anchor=(1.15, 1))

    plt.tight_layout()
    plt.show()


# ─── Génération des graphiques ────────────────────────────────────────────────
for factor_col, (factor_label, order) in factors.items():
    # ─── Part modale en distance ──────────────────────────────────────────────
    plot_modal_share_stacked(
        df_users, factor_col, factor_label, order,
        metric_prefix="share_dist",
        metric_title="Part modale en distance (%)"
    )

    # ─── Part modale en nombre de legs ────────────────────────────────────────
    plot_modal_share_stacked(
        df_users, factor_col, factor_label, order,
        metric_prefix="share_legs",
        metric_title="Part modale en nombre de legs (%)"
    )

In [ ]:
# ─── Vérification des valeurs de purpose_group ───────────────────────────────
print("── Valeurs purpose_group ────────────────────────────")
print(legs_GG_walk_regular["purpose_group"].value_counts(dropna=False))
print(f"\nTotal legs : {len(legs_GG_walk_regular)}")

In [ ]:
# ─── Distribution purpose_group by socio-demographic group ───────────────────
df_purpose = legs_GG_walk_regular.dropna(subset=["purpose_group"]).copy()

# ─── Mapping à la volée ───────────────────────────────────────────────────────
df_purpose["gdr"]            = df_purpose["gdr"].map(gender_fr_to_en)
df_purpose["age_fr_grouped"] = df_purpose["age_fr_grouped"].map(age_fr_to_en)
df_purpose["tp_level"]       = df_purpose["tp_level"].map(tp_level_map)

# ─── Socio-demographic factors ────────────────────────────────────────────────
factors_purpose = {
    'gdr'           : ('Gender',                      None),
    'age_fr_grouped': ('Age group',                   None),
    'income_class'  : ('Income per earner',           income_class_order),
    'has_car'       : ('Car ownership',               [labels_all_groups["no_car"], labels_all_groups["has_car"]]),
    'tp_level'      : ('PT subscription level',       tp_level_order),
}

print(f"Legs with purpose_group : {len(df_purpose):,} ({len(df_purpose)/len(legs_GG_walk_regular)*100:.1f}%)")
print(f"Legs without purpose_group : {legs_GG_walk_regular['purpose_group'].isna().sum():,} ({legs_GG_walk_regular['purpose_group'].isna().mean()*100:.1f}%)")

purpose_order = ["Constrained", "Semi-constrained", "Unconstrained", "Home", "Other"]

# ─── Text stats ───────────────────────────────────────────────────────────────
print(f"\n{'═'*70}")
print(f"  TRIP PURPOSE DISTRIBUTION BY SOCIO-DEMOGRAPHIC GROUP")
print(f"{'═'*70}")

for factor_col, (factor_label, order) in factors_purpose.items():
    _data = df_purpose.dropna(subset=[factor_col]).copy()

    if order is not None:
        order_filtered = [o for o in order if o in _data[factor_col].dropna().values]
        _data[factor_col] = pd.Categorical(_data[factor_col],
                                           categories=order_filtered, ordered=True)

    # ─── % by n_legs ──────────────────────────────────────────────────────────
    ct_legs = pd.crosstab(
        _data[factor_col],
        _data["purpose_group"],
        normalize="index"
    ) * 100
    ct_legs = ct_legs[[p for p in purpose_order if p in ct_legs.columns]]

    # ─── % by distance ────────────────────────────────────────────────────────
    ct_dist = _data.groupby([factor_col, "purpose_group"])["length_weighted"].sum()
    ct_dist = ct_dist.unstack(fill_value=0)
    ct_dist = ct_dist.div(ct_dist.sum(axis=1), axis=0) * 100
    ct_dist = ct_dist[[p for p in purpose_order if p in ct_dist.columns]]

    n_per_group = _data.groupby(factor_col)["leg_id"].count()

    print(f"\n── {factor_label} ──────────────────────────────────────")
    print(f"\n  By number of legs (%) :")
    ct_legs_display = ct_legs.copy()
    ct_legs_display.insert(0, "n_legs", n_per_group)
    print(ct_legs_display.round(1).to_string())

    print(f"\n  By weighted distance (%) :")
    ct_dist_display = ct_dist.copy()
    ct_dist_display.insert(0, "total_dist_m",
                           _data.groupby(factor_col)["length_weighted"].sum().round(0))
    print(ct_dist_display.round(1).to_string())

# ─── Charts ───────────────────────────────────────────────────────────────────
for metric, metric_label, n_title in [
    ("legs", "Share of walking trips (%)",    "trips"),
    ("dist", "Share of walking distance (%)", "m"),
]:
    print(f"\n── Trip purpose distribution — {metric_label} ─────────────────")

    
    n_factors  = len(factors_purpose)
    n_cols     = 3
    n_rows     = math.ceil(n_factors / n_cols)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 10))
    fig.patch.set_alpha(0)
    axes = axes.flatten()  # pour itérer facilement même si grille incomplète


    for ax, (factor_col, (factor_label, order)) in zip(axes, factors_purpose.items()):
        ax.patch.set_alpha(0)
        _data = df_purpose.dropna(subset=[factor_col]).copy()

        if order is not None:
            order_filtered = [o for o in order if o in _data[factor_col].dropna().values]
            _data[factor_col] = pd.Categorical(_data[factor_col],
                                               categories=order_filtered, ordered=True)
            categories = order_filtered
        else:
            categories = sorted(_data[factor_col].dropna().unique())

        if metric == "legs":
            ct = pd.crosstab(
                _data[factor_col],
                _data["purpose_group"],
                normalize="index"
            ) * 100
            n_label = _data.groupby(factor_col)["leg_id"].count()
        else:
            ct = _data.groupby([factor_col, "purpose_group"])["length_weighted"].sum()
            ct = ct.unstack(fill_value=0)
            ct = ct.div(ct.sum(axis=1), axis=0) * 100
            n_label = _data.groupby(factor_col)["length_weighted"].sum().round(0)

        ct = ct[[p for p in purpose_order if p in ct.columns]]
        ct = ct.reindex(categories)

        x      = np.arange(len(categories))
        bottom = np.zeros(len(categories))

        for purpose in purpose_order:
            if purpose not in ct.columns:
                continue
            values = ct[purpose].values
            ax.bar(x, values, bottom=bottom,
                   color=PURPOSE_GROUP_COLORS[purpose],
                   edgecolor="white", linewidth=0.5,
                   label=purpose)

            for i, (val, bot) in enumerate(zip(values, bottom)):
                if val > 5:
                    ax.text(x[i], bot + val / 2, f"{val:.1f}%",
                            ha='center', va='center', fontsize=14,
                            color='black') #fontweight='bold'
            bottom += values

        ax.set_xticks(x)
        # ─── Toujours utiliser le nombre de legs pour le label ───────────────────────
        n_legs_label = _data.groupby(factor_col)["leg_id"].count()

        ax.set_xticklabels(
            [f"{cat}\n(n={n_legs_label.get(cat, 0):,.0f} legs)"
            for cat in categories],
            rotation=30, ha='right', fontsize=14
        
        )
        ax.set_ylabel(metric_label, fontsize=14)
        ax.set_ylim(0, 105)
        ax.set_title(factor_label, fontsize=14, fontweight="bold")
        ax.grid(axis='y', alpha=0.3)

    for idx in range(n_factors, len(axes)):
        axes[idx].set_visible(False)

    handles = [
        plt.Rectangle((0, 0), 1, 1, color=PURPOSE_GROUP_COLORS[p], label=p)
        for p in purpose_order if p in PURPOSE_GROUP_COLORS
    ]
    fig.legend(handles=handles, loc="lower center", ncol=len(purpose_order),
               fontsize=16, bbox_to_anchor=(0.5, -0.05),
               title="Trip purpose group", title_fontsize=16)

    plt.tight_layout()
    plt.show()
    plt.close()